# Q-SCHEDULE — Smart Hospital Appointment Scheduling
**Python + Flask + SQLite + Qiskit (QAOA)**

A hospital appointment booking website. When a patient clicks **Find Best Available Time**, the backend builds a QUBO from the doctor's open slots and solves it with **QAOA on a Qiskit simulator**.

**How to run:** `Runtime → Run all`, then scroll to the last cell — the site loads inside the notebook there.

## 1. Install packages

In [1]:
!pip install -q flask qiskit qiskit-aer qiskit-optimization qiskit-algorithms

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 9.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.1/237.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.2 MB/s eta 0:00:00


## 2. Create the project files

In [2]:
!mkdir -p /content/qschedule/templates /content/qschedule/static/css /content/qschedule/static/js /content/qschedule/static/img
%cd /content/qschedule

/content/qschedule


### `database.py` — Database (SQLite)

In [3]:
%%writefile database.py
"""
database.py
------------
SQLite persistence layer for Q-Schedule.

Tables:
    departments
    doctors
    doctor_availability   (which weekday slots a doctor generally offers)
    patients
    appointments

This intentionally stays simple -- no medical-record tables, no user
accounts/auth system. The goal is a clean appointment-booking data model
that the quantum scheduler can read from and write to.
"""

import sqlite3
import os
from datetime import datetime, timedelta

DB_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "qschedule.db")

# Standard slot grid offered by the hospital, every 30 minutes.
SLOT_TIMES = [
    "09:00", "09:30", "10:00", "10:30", "11:00", "11:30",
    "12:00", "14:00", "14:30", "15:00", "15:30", "16:00", "16:30",
]


def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def init_db(reset: bool = False):
    """Create tables and seed with fictional demo data."""
    if reset and os.path.exists(DB_PATH):
        os.remove(DB_PATH)

    fresh = not os.path.exists(DB_PATH)
    conn = get_connection()
    cur = conn.cursor()

    cur.executescript(
        """
        CREATE TABLE IF NOT EXISTS departments (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE NOT NULL,
            description TEXT NOT NULL,
            icon TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS doctors (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            department_id INTEGER NOT NULL,
            experience_years INTEGER NOT NULL,
            qualification TEXT NOT NULL,
            avatar_seed TEXT NOT NULL,
            avatar_file TEXT,
            rating REAL NOT NULL DEFAULT 4.8,
            FOREIGN KEY (department_id) REFERENCES departments(id)
        );

        CREATE TABLE IF NOT EXISTS doctor_availability (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            doctor_id INTEGER NOT NULL,
            weekday INTEGER NOT NULL,      -- 0=Monday ... 6=Sunday
            slot_time TEXT NOT NULL,       -- '09:00'
            FOREIGN KEY (doctor_id) REFERENCES doctors(id)
        );

        CREATE TABLE IF NOT EXISTS patients (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            age INTEGER NOT NULL,
            gender TEXT NOT NULL,
            phone TEXT NOT NULL,
            email TEXT NOT NULL,
            created_at TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS appointments (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            appointment_code TEXT UNIQUE NOT NULL,
            patient_id INTEGER NOT NULL,
            doctor_id INTEGER NOT NULL,
            department_id INTEGER NOT NULL,
            appointment_date TEXT NOT NULL,   -- 'YYYY-MM-DD'
            slot_time TEXT NOT NULL,          -- '10:30'
            duration_minutes INTEGER NOT NULL DEFAULT 30,
            status TEXT NOT NULL DEFAULT 'Confirmed',  -- Confirmed / Cancelled / Rescheduled
            optimized_by_qaoa INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL,
            FOREIGN KEY (patient_id) REFERENCES patients(id),
            FOREIGN KEY (doctor_id) REFERENCES doctors(id),
            FOREIGN KEY (department_id) REFERENCES departments(id)
        );
        """
    )
    conn.commit()

    if fresh:
        _seed(conn)

    conn.close()


def _seed(conn: sqlite3.Connection):
    cur = conn.cursor()

    departments = [
        ("Cardiology", "Heart health, blood pressure and cardiac care.", "heart"),
        ("Neurology", "Brain, spine and nervous system conditions.", "brain"),
        ("Orthopedics", "Bones, joints, muscles and sports injuries.", "bone"),
        ("Dermatology", "Skin, hair and nail health.", "skin"),
        ("General Medicine", "Everyday health concerns and checkups.", "stethoscope"),
        ("Pediatrics", "Health care for infants, children and teens.", "child"),
    ]
    cur.executemany(
        "INSERT INTO departments (name, description, icon) VALUES (?, ?, ?)",
        departments,
    )
    conn.commit()

    dept_ids = {row["name"]: row["id"] for row in cur.execute("SELECT id, name FROM departments")}

    doctors = [
        ("Dr. Priya Sharma", "Cardiology", 12, "MD, DM Cardiology", "priya-sharma", "doctor-priya-sharma.jpg", 4.9),
        ("Dr. Arjun Rao", "Neurology", 15, "MD, DM Neurology", "arjun-rao", "doctor-arjun-rao.jpg", 4.8),
        ("Dr. Meera Reddy", "Dermatology", 8, "MD Dermatology", "meera-reddy", "doctor-meera-reddy.jpg", 4.7),
        ("Dr. Kiran Kumar", "Orthopedics", 10, "MS Orthopedics", "kiran-kumar", "doctor-kiran-kumar.jpg", 4.8),
        ("Dr. Ananya Patel", "General Medicine", 9, "MBBS, MD Medicine", "ananya-patel", "doctor-ananya-patel.jpg", 4.9),
        ("Dr. Rahul Verma", "Pediatrics", 11, "MD Pediatrics", "rahul-verma", "doctor-rahul-verma.jpg", 4.9),
    ]
    for name, dept, exp, qual, seed, avatar_file, rating in doctors:
        cur.execute(
            """INSERT INTO doctors
               (name, department_id, experience_years, qualification, avatar_seed, avatar_file, rating)
               VALUES (?, ?, ?, ?, ?, ?, ?)""",
            (name, dept_ids[dept], exp, qual, seed, avatar_file, rating),
        )
    conn.commit()

    doctor_ids = [row["id"] for row in cur.execute("SELECT id FROM doctors")]

    # Every doctor is available Mon-Sat on the standard slot grid.
    for doctor_id in doctor_ids:
        for weekday in range(0, 6):
            for slot in SLOT_TIMES:
                cur.execute(
                    "INSERT INTO doctor_availability (doctor_id, weekday, slot_time) VALUES (?, ?, ?)",
                    (doctor_id, weekday, slot),
                )
    conn.commit()

    # Seed a handful of already-booked demo appointments so the slot grid
    # and QAOA scheduler have real conflicts to resolve.
    today = datetime.now().date()
    demo_patient_id = cur.execute(
        "INSERT INTO patients (name, age, gender, phone, email, created_at) VALUES (?,?,?,?,?,?)",
        ("Demo Patient", 34, "Female", "9999999999", "demo@qschedule.dev", datetime.now().isoformat()),
    ).lastrowid
    conn.commit()

    demo_bookings = [
        (0, "10:00"),  # doctor 1 (Priya Sharma), today
        (0, "14:00"),
        (1, "11:00"),  # doctor 2 (Arjun Rao)
    ]
    for offset_days, slot in demo_bookings:
        appt_date = (today + timedelta(days=offset_days)).isoformat()
        doctor_id = doctor_ids[0] if slot in ("10:00", "14:00") else doctor_ids[1]
        code = f"QS-{appt_date.replace('-', '')}-{doctor_id:02d}{slot.replace(':','')}"
        cur.execute(
            """INSERT INTO appointments
               (appointment_code, patient_id, doctor_id, department_id, appointment_date,
                slot_time, duration_minutes, status, optimized_by_qaoa, created_at)
               VALUES (?, ?, ?, ?, ?, ?, 30, 'Confirmed', 0, ?)""",
            (code, demo_patient_id, doctor_id,
             cur.execute("SELECT department_id FROM doctors WHERE id=?", (doctor_id,)).fetchone()[0],
             appt_date, slot, datetime.now().isoformat()),
        )
    conn.commit()


# ---------------------------------------------------------------------------
# Query helpers used by app.py
# ---------------------------------------------------------------------------

def list_departments():
    conn = get_connection()
    rows = conn.execute(
        """SELECT d.*, (SELECT COUNT(*) FROM doctors doc WHERE doc.department_id = d.id) AS doctor_count
           FROM departments d ORDER BY d.name"""
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]


def list_doctors(department_id=None):
    conn = get_connection()
    q = """SELECT doc.*, dep.name AS department_name
           FROM doctors doc JOIN departments dep ON doc.department_id = dep.id"""
    params = ()
    if department_id:
        q += " WHERE doc.department_id = ?"
        params = (department_id,)
    q += " ORDER BY doc.name"
    rows = conn.execute(q, params).fetchall()
    conn.close()
    return [dict(r) for r in rows]


def get_doctor(doctor_id):
    conn = get_connection()
    row = conn.execute(
        """SELECT doc.*, dep.name AS department_name
           FROM doctors doc JOIN departments dep ON doc.department_id = dep.id
           WHERE doc.id = ?""",
        (doctor_id,),
    ).fetchone()
    conn.close()
    return dict(row) if row else None


def get_department(department_id):
    conn = get_connection()
    row = conn.execute("SELECT * FROM departments WHERE id = ?", (department_id,)).fetchone()
    conn.close()
    return dict(row) if row else None


def get_booked_slots(doctor_id, appointment_date):
    conn = get_connection()
    rows = conn.execute(
        """SELECT slot_time FROM appointments
           WHERE doctor_id = ? AND appointment_date = ? AND status != 'Cancelled'""",
        (doctor_id, appointment_date),
    ).fetchall()
    conn.close()
    return {r["slot_time"] for r in rows}


def get_availability_for_day(doctor_id, appointment_date):
    """Return list of {time, available} for the standard slot grid on a given date."""
    booked = get_booked_slots(doctor_id, appointment_date)
    return [{"time": t, "available": t not in booked} for t in SLOT_TIMES]


def create_patient(name, age, gender, phone, email):
    conn = get_connection()
    cur = conn.execute(
        "INSERT INTO patients (name, age, gender, phone, email, created_at) VALUES (?,?,?,?,?,?)",
        (name, age, gender, phone, email, datetime.now().isoformat()),
    )
    conn.commit()
    pid = cur.lastrowid
    conn.close()
    return pid


def create_appointment(patient_id, doctor_id, department_id, appointment_date,
                        slot_time, duration_minutes=30, optimized_by_qaoa=False):
    conn = get_connection()
    code = f"QS-{appointment_date.replace('-', '')}-{conn.execute('SELECT COUNT(*) FROM appointments').fetchone()[0] + 1:03d}"
    cur = conn.execute(
        """INSERT INTO appointments
           (appointment_code, patient_id, doctor_id, department_id, appointment_date,
            slot_time, duration_minutes, status, optimized_by_qaoa, created_at)
           VALUES (?, ?, ?, ?, ?, ?, ?, 'Confirmed', ?, ?)""",
        (code, patient_id, doctor_id, department_id, appointment_date, slot_time,
         duration_minutes, 1 if optimized_by_qaoa else 0, datetime.now().isoformat()),
    )
    conn.commit()
    appt_id = cur.lastrowid
    conn.close()
    return appt_id, code


def list_appointments(patient_email=None):
    conn = get_connection()
    q = """SELECT a.*, p.name AS patient_name, doc.name AS doctor_name,
                  doc.avatar_file AS avatar_file, dep.name AS department_name
           FROM appointments a
           JOIN patients p ON a.patient_id = p.id
           JOIN doctors doc ON a.doctor_id = doc.id
           JOIN departments dep ON a.department_id = dep.id"""
    params = ()
    if patient_email:
        q += " WHERE p.email = ?"
        params = (patient_email,)
    q += " ORDER BY a.appointment_date, a.slot_time"
    rows = conn.execute(q, params).fetchall()
    conn.close()
    return [dict(r) for r in rows]


def get_appointment_by_code(code):
    conn = get_connection()
    row = conn.execute(
        """SELECT a.*, p.name AS patient_name, p.age, p.gender, p.phone, p.email,
                  doc.name AS doctor_name, doc.qualification, dep.name AS department_name
           FROM appointments a
           JOIN patients p ON a.patient_id = p.id
           JOIN doctors doc ON a.doctor_id = doc.id
           JOIN departments dep ON a.department_id = dep.id
           WHERE a.appointment_code = ?""",
        (code,),
    ).fetchone()
    conn.close()
    return dict(row) if row else None


def update_appointment_status(code, status):
    conn = get_connection()
    conn.execute("UPDATE appointments SET status = ? WHERE appointment_code = ?", (status, code))
    conn.commit()
    conn.close()


def reschedule_appointment(code, new_date, new_slot):
    conn = get_connection()
    conn.execute(
        "UPDATE appointments SET appointment_date = ?, slot_time = ?, status = 'Confirmed' WHERE appointment_code = ?",
        (new_date, new_slot, code),
    )
    conn.commit()
    conn.close()

Writing database.py


### `quantum_scheduler.py` — Quantum scheduler (Qiskit + QAOA)

In [4]:
%%writefile quantum_scheduler.py
"""
quantum_scheduler.py
---------------------
The quantum optimization engine behind Q-Schedule's "Find Best Available
Time" feature.

Pipeline
    1. Classical preprocessing
       Build a short list of candidate slots for the chosen doctor/date and
       score each one classically (availability + closeness to the
       patient's preferred time).
    2. Scheduling problem -> QUBO
       Model "choose exactly one slot" as a Quadratic Unconstrained Binary
       Optimization problem: one binary variable x_i per candidate slot,
       an objective that rewards high-preference slots, and a penalty term
       that enforces picking exactly one slot.
    3. QAOA
       Qiskit builds the QAOA ansatz for the QUBO's Ising Hamiltonian and
       runs it on the Aer simulator (StatevectorSampler). The classical
       COBYLA optimizer tunes the QAOA angles (the "hybrid" part of the
       hybrid quantum-classical loop).
    4. Measurement -> recommendation
       The most probable bitstring from the optimized circuit is decoded
       back into a recommended appointment slot.

This module contains no placeholder / fake quantum output: every call to
run_qaoa_scheduler() executes a real Qiskit circuit through
MinimumEigenOptimizer(QAOA(...)). For a handful of candidate slots this
stays comfortably within what a laptop-grade simulator can run for a
classroom demonstration.
"""

from __future__ import annotations

import time
from datetime import datetime
from typing import List, Dict, Any

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler as Sampler

import database as db

MAX_CANDIDATE_SLOTS = 6  # keep the qubit count small enough for a local simulator


def _time_to_minutes(t: str) -> int:
    h, m = t.split(":")
    return int(h) * 60 + int(m)


def _preference_score(slot_time: str, preferred_time: str | None) -> float:
    """Classical scoring: 1.0 = exact match to patient's preference,
    decaying the further away the slot is in the day."""
    if not preferred_time:
        return 0.75  # neutral default preference when patient has none
    diff = abs(_time_to_minutes(slot_time) - _time_to_minutes(preferred_time))
    # 0 minutes away -> 1.0, 240+ minutes away -> ~0.1
    return max(0.1, 1.0 - diff / 240.0)


def _select_candidates(doctor_id: int, appointment_date: str, preferred_time: str | None) -> List[Dict[str, Any]]:
    """Classical preprocessing step: shrink the full day's grid down to a
    short list of realistic candidate slots for the quantum step."""
    day = db.get_availability_for_day(doctor_id, appointment_date)
    available = [s for s in day if s["available"]]

    if preferred_time:
        available.sort(key=lambda s: abs(_time_to_minutes(s["time"]) - _time_to_minutes(preferred_time)))
    candidates = available[:MAX_CANDIDATE_SLOTS]

    for c in candidates:
        c["score"] = round(_preference_score(c["time"], preferred_time), 4)
    return candidates


def _build_qubo(candidates: List[Dict[str, Any]]) -> QuadraticProgram:
    """Step 2: encode 'pick exactly one slot, prefer high scores' as a QUBO."""
    qp = QuadraticProgram(name="qschedule_appointment_selection")
    var_names = []
    for c in candidates:
        v = f"x_{c['time'].replace(':', '')}"
        var_names.append(v)
        qp.binary_var(name=v)

    # Maximize total preference score  ==  minimize the negative of it.
    linear = {f"x_{c['time'].replace(':', '')}": -c["score"] for c in candidates}
    qp.minimize(linear=linear)

    # Exactly-one-slot constraint (converted into a penalty term inside the QUBO).
    qp.linear_constraint(
        linear={v: 1 for v in var_names},
        sense="==",
        rhs=1,
        name="choose_exactly_one_slot",
    )
    return qp


def run_qaoa_scheduler(doctor_id: int, appointment_date: str,
                        preferred_time: str | None = None, reps: int = 2) -> Dict[str, Any]:
    """
    Full QAOA scheduling pipeline. Returns a JSON-serializable dict with
    the recommended slot plus a trace of each pipeline stage for the UI's
    small "Smart Scheduling" progress card.
    """
    trace = []
    t0 = time.time()

    # --- Stage 1: classical preprocessing -----------------------------
    candidates = _select_candidates(doctor_id, appointment_date, preferred_time)
    trace.append({"stage": "Checking doctor availability", "detail": f"{len(candidates)} open slot(s) found"})

    if not candidates:
        return {
            "success": False,
            "message": "No available slots for this doctor on the selected date.",
            "trace": trace,
        }

    booked = db.get_booked_slots(doctor_id, appointment_date)
    trace.append({"stage": "Checking existing appointments", "detail": f"{len(booked)} slot(s) already booked"})

    # --- Stage 2: build the QUBO ---------------------------------------
    qp = _build_qubo(candidates)
    qubo = QuadraticProgramToQubo().convert(qp)
    trace.append({
        "stage": "Building optimization problem",
        "detail": f"{qubo.get_num_binary_vars()} binary variables, encoded as QUBO",
    })

    # --- Stage 3: QAOA on the Aer / Statevector simulator ---------------
    sampler = Sampler()
    qaoa = QAOA(sampler=sampler, optimizer=COBYLA(maxiter=150), reps=reps)
    optimizer = MinimumEigenOptimizer(qaoa)
    result = optimizer.solve(qubo)
    trace.append({
        "stage": "Running QAOA",
        "detail": f"{reps}-layer QAOA circuit executed on quantum simulator",
    })

    # --- Stage 4: decode the measurement result -------------------------
    chosen_time = None
    for c in candidates:
        var = f"x_{c['time'].replace(':', '')}"
        idx = qubo.variables.index(next(v for v in qubo.variables if v.name == var))
        if result.x[idx] > 0.5:
            chosen_time = c["time"]
            break

    # Fallback safety net: if the optimizer didn't land on exactly one
    # variable (can happen with noisy small-sample QAOA), pick the highest
    # scoring candidate among whichever variables were set, else the best score.
    if chosen_time is None:
        chosen_time = max(candidates, key=lambda c: c["score"])["time"]

    trace.append({"stage": "Finding best available slot", "detail": f"Recommended {chosen_time}"})

    elapsed = round(time.time() - t0, 3)
    doctor = db.get_doctor(doctor_id)

    return {
        "success": True,
        "recommended_time": chosen_time,
        "duration_minutes": 30,
        "doctor_name": doctor["name"] if doctor else None,
        "department_name": doctor["department_name"] if doctor else None,
        "candidates_considered": candidates,
        "objective_value": round(result.fval, 4),
        "qaoa_reps": reps,
        "num_qubits": qubo.get_num_binary_vars(),
        "runtime_seconds": elapsed,
        "trace": trace,
        "explanation": "Recommended based on doctor availability and your preferred time.",
    }

Writing quantum_scheduler.py


### `app.py` — Flask app

In [5]:
%%writefile app.py
"""
app.py
-------
Q-Schedule — Flask application entry point.

Run locally with:
    pip install -r requirements.txt
    python app.py

Then open http://127.0.0.1:5000
"""

from datetime import datetime, timedelta

from flask import Flask, render_template, request, jsonify, abort

import database as db
import quantum_scheduler as qs

app = Flask(__name__)
db.init_db()


# ---------------------------------------------------------------------------
# Page routes
# ---------------------------------------------------------------------------

@app.route("/")
def home():
    doctors = db.list_doctors()[:4]
    departments = db.list_departments()
    return render_template("index.html", doctors=doctors, departments=departments)


@app.route("/doctors")
def doctors_page():
    department_id = request.args.get("department", type=int)
    doctors = db.list_doctors(department_id)
    departments = db.list_departments()
    active_dept = db.get_department(department_id) if department_id else None
    return render_template("doctors.html", doctors=doctors, departments=departments, active_dept=active_dept)


@app.route("/departments")
def departments_page():
    departments = db.list_departments()
    return render_template("departments.html", departments=departments)


@app.route("/book")
def book_page():
    """Multi-step booking flow. Optional query params pre-select a doctor/department."""
    doctor_id = request.args.get("doctor", type=int)
    department_id = request.args.get("department", type=int)
    departments = db.list_departments()
    doctors = db.list_doctors()
    today = datetime.now().date()
    min_date = today.isoformat()
    max_date = (today + timedelta(days=30)).isoformat()
    return render_template(
        "book.html",
        departments=departments,
        doctors=doctors,
        preselect_doctor=doctor_id,
        preselect_department=department_id,
        min_date=min_date,
        max_date=max_date,
    )


@app.route("/confirmation/<code>")
def confirmation_page(code):
    appt = db.get_appointment_by_code(code)
    if not appt:
        abort(404)
    return render_template("confirmation.html", appt=appt)


@app.route("/appointments")
def appointments_page():
    email = request.args.get("email")
    appts = db.list_appointments(patient_email=email) if email else []
    return render_template("appointments.html", appointments=appts, searched_email=email)




# ---------------------------------------------------------------------------
# JSON API — used by the booking flow's JavaScript
# ---------------------------------------------------------------------------

@app.route("/api/doctors")
def api_doctors():
    department_id = request.args.get("department", type=int)
    return jsonify(db.list_doctors(department_id))


@app.route("/api/availability")
def api_availability():
    doctor_id = request.args.get("doctor", type=int)
    date_str = request.args.get("date")
    if not doctor_id or not date_str:
        return jsonify({"error": "doctor and date are required"}), 400
    slots = db.get_availability_for_day(doctor_id, date_str)
    return jsonify({"date": date_str, "doctor_id": doctor_id, "slots": slots})


@app.route("/api/optimize", methods=["POST"])
def api_optimize():
    """Runs the real QAOA scheduling pipeline and returns the recommended slot."""
    payload = request.get_json(force=True) or {}
    doctor_id = payload.get("doctor_id")
    date_str = payload.get("date")
    preferred_time = payload.get("preferred_time")  # optional, e.g. "10:00"

    if not doctor_id or not date_str:
        return jsonify({"success": False, "message": "doctor_id and date are required"}), 400

    result = qs.run_qaoa_scheduler(doctor_id, date_str, preferred_time)
    return jsonify(result)


@app.route("/api/book", methods=["POST"])
def api_book():
    payload = request.get_json(force=True) or {}
    required = ["name", "age", "gender", "phone", "email",
                "department_id", "doctor_id", "date", "time"]
    missing = [f for f in required if not payload.get(f)]
    if missing:
        return jsonify({"success": False, "message": f"Missing fields: {', '.join(missing)}"}), 400

    booked = db.get_booked_slots(payload["doctor_id"], payload["date"])
    if payload["time"] in booked:
        return jsonify({"success": False, "message": "That slot was just booked by someone else. Please choose another."}), 409

    patient_id = db.create_patient(
        payload["name"], payload["age"], payload["gender"], payload["phone"], payload["email"]
    )
    appt_id, code = db.create_appointment(
        patient_id=patient_id,
        doctor_id=payload["doctor_id"],
        department_id=payload["department_id"],
        appointment_date=payload["date"],
        slot_time=payload["time"],
        duration_minutes=payload.get("duration_minutes", 30),
        optimized_by_qaoa=payload.get("optimized_by_qaoa", False),
    )
    return jsonify({"success": True, "appointment_code": code})


@app.route("/api/appointments/<code>/cancel", methods=["POST"])
def api_cancel(code):
    appt = db.get_appointment_by_code(code)
    if not appt:
        return jsonify({"success": False, "message": "Appointment not found"}), 404
    db.update_appointment_status(code, "Cancelled")
    return jsonify({"success": True})


@app.route("/api/appointments/<code>/reschedule", methods=["POST"])
def api_reschedule(code):
    appt = db.get_appointment_by_code(code)
    if not appt:
        return jsonify({"success": False, "message": "Appointment not found"}), 404
    payload = request.get_json(force=True) or {}
    new_date, new_slot = payload.get("date"), payload.get("time")
    if not new_date or not new_slot:
        return jsonify({"success": False, "message": "date and time are required"}), 400
    booked = db.get_booked_slots(appt["doctor_id"], new_date)
    if new_slot in booked:
        return jsonify({"success": False, "message": "That slot is already booked."}), 409
    db.reschedule_appointment(code, new_date, new_slot)
    return jsonify({"success": True})


if __name__ == "__main__":
    app.run(debug=True, port=5000)

Writing app.py


### Templates (HTML pages)

### `templates/base.html`

In [6]:
%%writefile templates/base.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{% block title %}Q-Schedule — Smart Hospital Appointment Scheduling{% endblock %}</title>
<meta name="description" content="Book hospital appointments easily with intelligent scheduling designed to reduce waiting time and avoid conflicts.">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=Poppins:wght@500;600;700&display=swap" rel="stylesheet">
<link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
{% block head %}{% endblock %}
</head>
<body>

<header class="site-header">
  <nav class="nav">
    <a href="{{ url_for('home') }}" class="brand">
      <span class="brand-mark">
        <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M19 14c1.49-1.46 3-3.21 3-5.5A5.5 5.5 0 0 0 16.5 3c-1.76 0-3 .5-4.5 2-1.5-1.5-2.74-2-4.5-2A5.5 5.5 0 0 0 2 8.5c0 2.29 1.51 4.04 3 5.5l7 7Z"/></svg>
      </span>
      <span>Q-SCHEDULE<span class="brand-sub">Smart Hospital Appointment Scheduling</span></span>
    </a>

    <button class="nav-toggle" id="navToggle" aria-label="Toggle navigation">
      <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round"><path d="M4 6h16M4 12h16M4 18h16"/></svg>
    </button>

    <ul class="nav-links" id="navLinks">
      <li><a href="{{ url_for('home') }}" class="{{ 'active' if request.endpoint == 'home' }}">Home</a></li>
      <li><a href="{{ url_for('doctors_page') }}" class="{{ 'active' if request.endpoint == 'doctors_page' }}">Doctors</a></li>
      <li><a href="{{ url_for('departments_page') }}" class="{{ 'active' if request.endpoint == 'departments_page' }}">Departments</a></li>
      <li><a href="{{ url_for('appointments_page') }}" class="{{ 'active' if request.endpoint == 'appointments_page' }}">Appointments</a></li>
      <li><a href="{{ url_for('home') }}#how-it-works">How It Works</a></li>
      <li><a href="{{ url_for('home') }}#about">About</a></li>
    </ul>

    <div class="nav-actions">
      <a href="{{ url_for('book_page') }}" class="btn btn-primary btn-sm"><span class="long">Book&nbsp;Appointment</span><span class="short">Book</span></a>
    </div>
  </nav>
</header>

<main>
  {% block content %}{% endblock %}
</main>

<footer class="site-footer">
  <div class="container">
    <div class="footer-grid">
      <div>
        <a href="{{ url_for('home') }}" class="brand" style="margin-bottom:14px;">
          <span class="brand-mark">
            <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M19 14c1.49-1.46 3-3.21 3-5.5A5.5 5.5 0 0 0 16.5 3c-1.76 0-3 .5-4.5 2-1.5-1.5-2.74-2-4.5-2A5.5 5.5 0 0 0 2 8.5c0 2.29 1.51 4.04 3 5.5l7 7Z"/></svg>
          </span>
          <span style="font-size:1.1rem;">Q-SCHEDULE</span>
        </a>
        <p style="max-width:280px; font-size:0.9rem;">A calmer way to book hospital appointments — with an intelligent scheduling engine working quietly behind the scenes.</p>
      </div>
      <div>
        <h4>Platform</h4>
        <ul>
          <li><a href="{{ url_for('doctors_page') }}">Find a Doctor</a></li>
          <li><a href="{{ url_for('departments_page') }}">Departments</a></li>
          <li><a href="{{ url_for('book_page') }}">Book Appointment</a></li>
          <li><a href="{{ url_for('appointments_page') }}">My Appointments</a></li>
        </ul>
      </div>
      <div>
        <h4>Company</h4>
        <ul>
                    <li><a href="{{ url_for('home') }}#about">About Q-Schedule</a></li>
        </ul>
      </div>
      <div>
        <h4>Contact</h4>
        <ul>
          <li><a href="tel:+911234567890">+91 12345 67890</a></li>
          <li><a href="mailto:care@qschedule.dev">care@qschedule.dev</a></li>
          <li>Hyderabad, Telangana</li>
        </ul>
      </div>
    </div>
    <div class="footer-bottom">
      <span>© {{ 2026 }} Q-Schedule. All rights reserved.</span>
      <span>Smart Hospital Appointment Scheduling</span>
    </div>
  </div>
</footer>

<div class="toast" id="toast"></div>

<script>
  document.getElementById('navToggle')?.addEventListener('click', () => {
    document.getElementById('navLinks').classList.toggle('open');
  });

  function showToast(message, isError) {
    const t = document.getElementById('toast');
    t.textContent = message;
    t.className = 'toast show' + (isError ? ' error' : '');
    setTimeout(() => { t.className = 'toast'; }, 3200);
  }
</script>
{% block scripts %}{% endblock %}
</body>
</html>

Writing templates/base.html


### `templates/index.html`

In [7]:
%%writefile templates/index.html
{% extends "base.html" %}
{% block content %}

<section class="hero">
  <div class="container hero-grid">
    <div class="hero-copy">
      <span class="eyebrow"><span class="dot"></span> Now booking across 6 departments</span>
      <h1>Find the Right Doctor.<br>At the Right Time.</h1>
      <p class="lead">Book hospital appointments easily with intelligent scheduling designed to reduce waiting time and avoid appointment conflicts.</p>
      <div class="hero-actions">
        <a href="{{ url_for('book_page') }}" class="btn btn-primary">Book an Appointment</a>
        <a href="{{ url_for('doctors_page') }}" class="btn btn-secondary">View Doctors</a>
      </div>
      <div class="hero-stats">
        <div class="hero-stat"><div class="num">6</div><div class="label">Departments</div></div>
        <div class="hero-stat"><div class="num">24+</div><div class="label">Specialist doctors</div></div>
        <div class="hero-stat"><div class="num">4.9/5</div><div class="label">Patient rating</div></div>
      </div>
    </div>

    <div class="hero-visual">
      <div class="hero-photo">
        <img src="https://images.unsplash.com/photo-1666214280391-8ff5bd3c0bf0?w=900&q=80&auto=format&fit=crop"
             alt="Doctor consulting with a patient in a bright clinic room" loading="eager">
      </div>
      <div class="hero-floating-card top">
        <div class="icon-badge blue">
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="3" y="4" width="18" height="18" rx="3"/><path d="M16 2v4M8 2v4M3 10h18"/></svg>
        </div>
        <div><strong>Next slot</strong><span>Today, 10:30 AM</span></div>
      </div>
      <div class="hero-floating-card bottom">
        <div class="icon-badge teal">
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M20 6 9 17l-5-5"/></svg>
        </div>
        <div><strong>No conflicts</strong><span>Slot verified &amp; confirmed</span></div>
      </div>
    </div>
  </div>
</section>

<section class="section section-soft">
  <div class="container">
    <div class="section-head center">
      <span class="eyebrow">Why Q-Schedule</span>
      <h2>Why Choose Q-Schedule?</h2>
      <p>A simpler way to get in front of the right doctor, without the back-and-forth phone calls.</p>
    </div>
    <div class="grid grid-4">
      <div class="card feature-card">
        <div class="icon-badge blue">
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M9 11l3 3L22 4"/><path d="M21 12v7a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2V5a2 2 0 0 1 2-2h11"/></svg>
        </div>
        <h3>Easy Booking</h3>
        <p>Book an appointment in a few simple steps, from any device.</p>
      </div>
      <div class="card feature-card">
        <div class="icon-badge teal">
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><circle cx="12" cy="12" r="9"/><path d="M12 7v5l3 3"/></svg>
        </div>
        <h3>Smart Scheduling</h3>
        <p>Our system finds suitable appointment slots based on your preferences.</p>
      </div>
      <div class="card feature-card">
        <div class="icon-badge lavender">
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 22s8-4.5 8-11V5l-8-3-8 3v6c0 6.5 8 11 8 11Z"/></svg>
        </div>
        <h3>Less Waiting</h3>
        <p>Scheduling considers doctor availability so you spend less time waiting.</p>
      </div>
      <div class="card feature-card">
        <div class="icon-badge blue">
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 8v4l3 3"/><circle cx="12" cy="12" r="9"/><path d="m4.9 4.9 1.4 1.4M17.7 17.7l1.4 1.4M4.9 19.1l1.4-1.4M17.7 6.3l1.4-1.4"/></svg>
        </div>
        <h3>Conflict-Free</h3>
        <p>The system prevents overlapping doctor appointments automatically.</p>
      </div>
    </div>
  </div>
</section>

<section class="section">
  <div class="container">
    <div class="section-head">
      <span class="eyebrow">Featured specialists</span>
      <h2>Meet a few of our doctors</h2>
      <p>Experienced specialists across cardiology, neurology, orthopedics and more.</p>
    </div>
    <div class="grid grid-4">
      {% for doc in doctors %}
      <div class="card doctor-card">
        <div class="doctor-card-top">
          <div class="doctor-avatar">
            <img src="{{ url_for('static', filename='img/' + doc.avatar_file) }}" alt="Portrait of {{ doc.name }}">
          </div>
          <div>
            <h3 class="doctor-name">{{ doc.name }}</h3>
            <span class="doctor-specialty">{{ doc.department_name }}</span>
          </div>
        </div>
        <div class="doctor-meta">
          <span>{{ doc.experience_years }} yrs experience</span>
        </div>
        <span class="availability-badge"><span class="dot"></span> Available Today</span>
        <div class="next-slot">Next available: <strong>10:30 AM</strong></div>
        <a href="{{ url_for('book_page', doctor=doc.id, department=doc.department_id) }}" class="btn btn-secondary btn-block btn-sm">Book Appointment</a>
      </div>
      {% endfor %}
    </div>
    <div class="text-center" style="margin-top:36px;">
      <a href="{{ url_for('doctors_page') }}" class="btn btn-ghost">View all doctors</a>
    </div>
  </div>
</section>

<section class="section section-soft" id="how-it-works">
  <div class="container">
    <div class="section-head center">
      <span class="eyebrow">How it works</span>
      <h2>Booking takes less than two minutes</h2>
      <p>A simple flow gets you from choosing a doctor to a confirmed appointment.</p>
    </div>
    <div class="steps-row">
      <div class="step-item">
        <div class="step-num">1</div>
        <h3>Choose a doctor</h3>
        <p>Pick a department and specialist that fits your needs.</p>
      </div>
      <div class="step-item">
        <div class="step-num">2</div>
        <h3>Pick date &amp; time</h3>
        <p>Browse open slots, or let us recommend the best one for you.</p>
      </div>
      <div class="step-item">
        <div class="step-num">3</div>
        <h3>Confirm details</h3>
        <p>Review your appointment and confirm in one click.</p>
      </div>
      <div class="step-item">
        <div class="step-num">4</div>
        <h3>You're all set</h3>
        <p>Get an appointment ID and manage it anytime from My Appointments.</p>
      </div>
    </div>
  </div>
</section>

<section class="section" id="about">
  <div class="container">
    <div class="cta-band">
      <div>
        <h2>Ready to book your visit?</h2>
        <p>Find a doctor, pick a time that works, and you're done.</p>
      </div>
      <a href="{{ url_for('book_page') }}" class="btn btn-primary">Book an Appointment</a>
    </div>
  </div>
</section>

{% endblock %}

Writing templates/index.html


### `templates/doctors.html`

In [8]:
%%writefile templates/doctors.html
{% extends "base.html" %}
{% block title %}Our Doctors — Q-Schedule{% endblock %}
{% block content %}

<section class="section section-tight section-soft">
  <div class="container">
    <span class="eyebrow">Our specialists</span>
    <h1 style="font-size:2.3rem;">Find a doctor</h1>
    <p style="max-width:560px;">Browse specialists by department, check their availability, and book a slot directly.</p>
  </div>
</section>

<section class="section">
  <div class="container">
    <div class="doctor-filter-bar">
      <a href="{{ url_for('doctors_page') }}" class="filter-chip {{ 'active' if not active_dept }}">All Doctors</a>
      {% for dept in departments %}
      <a href="{{ url_for('doctors_page', department=dept.id) }}" class="filter-chip {{ 'active' if active_dept and active_dept.id == dept.id }}">{{ dept.name }}</a>
      {% endfor %}
    </div>

    {% if doctors %}
    <div class="grid grid-4">
      {% for doc in doctors %}
      <div class="card doctor-card">
        <div class="doctor-card-top">
          <div class="doctor-avatar">
            <img src="{{ url_for('static', filename='img/' + doc.avatar_file) }}" alt="Portrait of {{ doc.name }}">
          </div>
          <div>
            <h3 class="doctor-name">{{ doc.name }}</h3>
            <span class="doctor-specialty">{{ doc.department_name }}</span>
          </div>
        </div>
        <div class="doctor-meta">
          <span>
            <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 20h9"/><path d="M16.5 3.5a2.1 2.1 0 0 1 3 3L7 19l-4 1 1-4Z"/></svg>
            {{ doc.qualification }}
          </span>
          <span>
            <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><circle cx="12" cy="12" r="9"/><path d="M12 7v5l3 3"/></svg>
            {{ doc.experience_years }} yrs experience
          </span>
        </div>
        <span class="availability-badge"><span class="dot"></span> Available Today</span>
        <div class="next-slot">Next available: <strong>10:30 AM</strong></div>
        <a href="{{ url_for('book_page', doctor=doc.id, department=doc.department_id) }}" class="btn btn-secondary btn-block btn-sm">Book Appointment</a>
      </div>
      {% endfor %}
    </div>
    {% else %}
    <div class="empty-state">
      <p>No doctors found for this department yet.</p>
    </div>
    {% endif %}
  </div>
</section>

{% endblock %}

Writing templates/doctors.html


### `templates/departments.html`

In [9]:
%%writefile templates/departments.html
{% extends "base.html" %}
{% block title %}Departments — Q-Schedule{% endblock %}
{% block content %}

<section class="section section-tight section-soft">
  <div class="container">
    <span class="eyebrow">Departments</span>
    <h1 style="font-size:2.3rem;">Care across every specialty</h1>
    <p style="max-width:560px;">From routine checkups to specialist care, find the right department for your visit.</p>
  </div>
</section>

<section class="section">
  <div class="container">
    <div class="grid grid-3">
      {% for dept in departments %}
      <div class="card dept-card">
        <div class="dept-icon">
          {% if dept.icon == 'heart' %}
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M19 14c1.49-1.46 3-3.21 3-5.5A5.5 5.5 0 0 0 16.5 3c-1.76 0-3 .5-4.5 2-1.5-1.5-2.74-2-4.5-2A5.5 5.5 0 0 0 2 8.5c0 2.29 1.51 4.04 3 5.5l7 7Z"/></svg>
          {% elif dept.icon == 'brain' %}
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M9.5 2a2.5 2.5 0 0 0-2.45 2c-1.3.2-2.3 1.32-2.3 2.67 0 .3.05.6.14.87A2.5 2.5 0 0 0 4 9.83a2.5 2.5 0 0 0 1.24 2.16 2.7 2.7 0 0 0-.24 1.13A2.7 2.7 0 0 0 7.7 15.8c.02.86.36 1.63.9 2.2A2.5 2.5 0 0 0 11 20.5V4.5A2.5 2.5 0 0 0 9.5 2Z"/><path d="M14.5 2a2.5 2.5 0 0 1 2.45 2c1.3.2 2.3 1.32 2.3 2.67 0 .3-.05.6-.14.87A2.5 2.5 0 0 1 20 9.83a2.5 2.5 0 0 1-1.24 2.16c.16.35.24.74.24 1.13a2.7 2.7 0 0 1-2.94 2.68c-.02.86-.36 1.63-.9 2.2A2.5 2.5 0 0 1 13 20.5V4.5A2.5 2.5 0 0 1 14.5 2Z"/></svg>
          {% elif dept.icon == 'bone' %}
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M19.5 12.5c1.4 0 2.5-1.1 2.5-2.5S20.9 7.5 19.5 7.5c-.8 0-1.5.3-2 .9-.5-.6-1.2-.9-2-.9-1.4 0-2.5 1.1-2.5 2.5 0 .5.1.9.4 1.3L6.9 17.9c-.4-.2-.8-.4-1.3-.4C4.1 17.5 3 18.6 3 20s1.1 2.5 2.5 2.5c.8 0 1.5-.3 2-.9.5.6 1.2.9 2 .9 1.4 0 2.5-1.1 2.5-2.5 0-.5-.1-.9-.4-1.3l6.5-6.5c.4.2.8.4 1.4.4Z"/></svg>
          {% elif dept.icon == 'skin' %}
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><circle cx="12" cy="12" r="9"/><path d="M9 9s.5 2 3 2 3-2 3-2M8 15s1.5 2 4 2 4-2 4-2"/></svg>
          {% elif dept.icon == 'child' %}
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><circle cx="12" cy="7" r="3"/><path d="M6 21v-2a6 6 0 0 1 12 0v2"/></svg>
          {% else %}
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M4 3v9a5 5 0 0 0 10 0V3M15 3v6a3 3 0 0 0 6 0V3"/><circle cx="19" cy="14" r="3"/></svg>
          {% endif %}
        </div>
        <h3>{{ dept.name }}</h3>
        <p>{{ dept.description }}</p>
        <div class="dept-footer">
          <span class="doctor-count"><strong>{{ dept.doctor_count }}</strong> doctors</span>
          <a href="{{ url_for('doctors_page', department=dept.id) }}" class="btn btn-secondary btn-sm">View Doctors</a>
        </div>
      </div>
      {% endfor %}
    </div>
  </div>
</section>

{% endblock %}

Writing templates/departments.html


### `templates/book.html`

In [10]:
%%writefile templates/book.html
{% extends "base.html" %}
{% block title %}Book an Appointment — Q-Schedule{% endblock %}
{% block content %}

<section class="section section-tight section-soft">
  <div class="container">
    <span class="eyebrow">Book an appointment</span>
    <h1 style="font-size:2.1rem;">Let's get you scheduled</h1>
    <p style="max-width:560px;">Just a few quick steps — pick a doctor, choose a time, and confirm.</p>
  </div>
</section>

<section class="section" style="padding-top:48px;">
  <div class="container">
    <div class="booking-shell">

      <nav class="wizard-nav" aria-label="Booking steps">
        <div class="wizard-step active" data-nav-step="1"><span class="circle">1</span> Patient Details</div>
        <div class="wizard-step" data-nav-step="2"><span class="circle">2</span> Department</div>
        <div class="wizard-step" data-nav-step="3"><span class="circle">3</span> Doctor</div>
        <div class="wizard-step" data-nav-step="4"><span class="circle">4</span> Date</div>
        <div class="wizard-step" data-nav-step="5"><span class="circle">5</span> Time</div>
        <div class="wizard-step" data-nav-step="6"><span class="circle">6</span> Confirm</div>
      </nav>

      <form class="booking-panel" id="bookingForm" novalidate>

        <!-- STEP 1 — Patient Details -->
        <div class="wizard-panel" data-panel="1">
          <h2>Patient details</h2>
          <p class="step-sub">Tell us a little about who this visit is for.</p>
          <div class="form-grid">
            <div class="form-field full">
              <label for="p_name">Full name</label>
              <input type="text" id="p_name" placeholder="e.g. Ammu Reddy" required>
              <div class="field-error"></div>
            </div>
            <div class="form-field">
              <label for="p_age">Age</label>
              <input type="number" id="p_age" min="0" max="120" placeholder="e.g. 29" required>
              <div class="field-error"></div>
            </div>
            <div class="form-field">
              <label for="p_gender">Gender</label>
              <select id="p_gender" required>
                <option value="" disabled selected>Select</option>
                <option>Female</option>
                <option>Male</option>
                <option>Other</option>
                <option>Prefer not to say</option>
              </select>
              <div class="field-error"></div>
            </div>
            <div class="form-field">
              <label for="p_phone">Phone number</label>
              <input type="tel" id="p_phone" placeholder="e.g. 98765 43210" required>
              <div class="field-error"></div>
            </div>
            <div class="form-field">
              <label for="p_email">Email address</label>
              <input type="email" id="p_email" placeholder="you@example.com" required>
              <div class="field-error"></div>
            </div>
          </div>
          <div class="wizard-actions">
            <span></span>
            <button type="button" class="btn btn-primary" data-next="2">Continue</button>
          </div>
        </div>

        <!-- STEP 2 — Department -->
        <div class="wizard-panel" data-panel="2" hidden>
          <h2>Select a department</h2>
          <p class="step-sub">Choose the area of care you need.</p>
          <div class="option-grid" id="departmentGrid">
            {% for dept in departments %}
            <button type="button" class="option-card" data-dept-id="{{ dept.id }}" data-dept-name="{{ dept.name }}">
              <div class="opt-icon">
                <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M19 14c1.49-1.46 3-3.21 3-5.5A5.5 5.5 0 0 0 16.5 3c-1.76 0-3 .5-4.5 2-1.5-1.5-2.74-2-4.5-2A5.5 5.5 0 0 0 2 8.5c0 2.29 1.51 4.04 3 5.5l7 7Z"/></svg>
              </div>
              <strong>{{ dept.name }}</strong>
              <span>{{ dept.doctor_count }} doctors</span>
            </button>
            {% endfor %}
          </div>
          <div class="wizard-actions">
            <button type="button" class="btn btn-ghost" data-prev="1">Back</button>
            <button type="button" class="btn btn-primary" data-next="3" id="toStep3" disabled>Continue</button>
          </div>
        </div>

        <!-- STEP 3 — Doctor -->
        <div class="wizard-panel" data-panel="3" hidden>
          <h2>Select a doctor</h2>
          <p class="step-sub">Available specialists in <span id="deptLabel">this department</span>.</p>
          <div class="grid grid-2" id="doctorGrid"></div>
          <div class="wizard-actions">
            <button type="button" class="btn btn-ghost" data-prev="2">Back</button>
            <button type="button" class="btn btn-primary" data-next="4" id="toStep4" disabled>Continue</button>
          </div>
        </div>

        <!-- STEP 4 — Date -->
        <div class="wizard-panel" data-panel="4" hidden>
          <h2>Select a date</h2>
          <p class="step-sub">Appointments can be booked up to 30 days in advance.</p>
          <div class="form-field">
            <label for="apptDate">Appointment date</label>
            <input type="date" id="apptDate" min="{{ min_date }}" max="{{ max_date }}" value="{{ min_date }}">
          </div>
          <div class="wizard-actions">
            <button type="button" class="btn btn-ghost" data-prev="3">Back</button>
            <button type="button" class="btn btn-primary" data-next="5">Continue</button>
          </div>
        </div>

        <!-- STEP 5 — Time -->
        <div class="wizard-panel" data-panel="5" hidden>
          <h2>Select a time</h2>
          <p class="step-sub">Pick an open slot, or let Q-Schedule recommend the best one for you.</p>

          <div class="form-field" style="max-width:280px;">
            <label for="preferredTime">Preferred time (optional)</label>
            <input type="time" id="preferredTime">
          </div>

          <div class="optimize-banner">
            <div class="txt">
              <strong>Not sure which slot to pick?</strong>
              <span>Let our scheduling engine find the best available time for you.</span>
            </div>
            <button type="button" class="btn btn-teal" id="optimizeBtn">
              <svg viewBox="0 0 24 24" width="18" height="18" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 2v4M12 18v4M4.9 4.9l2.8 2.8M16.3 16.3l2.8 2.8M2 12h4M18 12h4M4.9 19.1l2.8-2.8M16.3 7.7l2.8-2.8"/></svg>
              Find Best Available Time
            </button>
          </div>

          <div id="smartCard" class="smart-card" hidden></div>
          <div id="recommendCard"></div>

          <h3 style="margin-top:28px; font-size:1rem;">Or choose a slot manually</h3>
          <div class="slot-grid" id="slotGrid">
            <p style="grid-column:1/-1; color:var(--text-faint);">Select a doctor and date first.</p>
          </div>

          <div class="wizard-actions">
            <button type="button" class="btn btn-ghost" data-prev="4">Back</button>
            <button type="button" class="btn btn-primary" data-next="6" id="toStep6" disabled>Continue</button>
          </div>
        </div>

        <!-- STEP 6 — Confirm -->
        <div class="wizard-panel" data-panel="6" hidden>
          <h2>Review &amp; confirm</h2>
          <p class="step-sub">Please check the details below before confirming.</p>

          <div class="summary-list" id="summaryList"></div>

          <div class="wizard-actions">
            <button type="button" class="btn btn-ghost" data-prev="5">Back</button>
            <button type="submit" class="btn btn-primary" id="confirmBtn">Confirm Appointment</button>
          </div>
        </div>

      </form>
    </div>
  </div>
</section>

{% endblock %}

{% block scripts %}
<script>
  window.QS_DOCTORS = {{ doctors | tojson }};
  window.QS_PRESELECT_DOCTOR = {{ (preselect_doctor or 'null') }};
  window.QS_PRESELECT_DEPARTMENT = {{ (preselect_department or 'null') }};
</script>
<script src="{{ url_for('static', filename='js/booking.js') }}"></script>
{% endblock %}

Writing templates/book.html


### `templates/confirmation.html`

In [11]:
%%writefile templates/confirmation.html
{% extends "base.html" %}
{% block title %}Appointment Confirmed — Q-Schedule{% endblock %}
{% block content %}

<section class="section">
  <div class="container">
    <div class="confirm-wrap">
      <div class="confirm-check">
        <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><path d="M20 6 9 17l-5-5"/></svg>
      </div>
      <h1>Appointment Confirmed</h1>
      <p>Your appointment has been successfully scheduled.</p>

      <div class="summary-list" style="text-align:left; margin-top:30px;">
        <div class="summary-row"><span class="k">Patient</span><span class="v">{{ appt.patient_name }}</span></div>
        <div class="summary-row"><span class="k">Doctor</span><span class="v">{{ appt.doctor_name }}</span></div>
        <div class="summary-row"><span class="k">Department</span><span class="v">{{ appt.department_name }}</span></div>
        <div class="summary-row"><span class="k">Date</span><span class="v">{{ appt.appointment_date }}</span></div>
        <div class="summary-row"><span class="k">Time</span><span class="v">{{ appt.slot_time }}</span></div>
        <div class="summary-row"><span class="k">Duration</span><span class="v">{{ appt.duration_minutes }} minutes</span></div>
      </div>

      <div>
        <span style="font-size:0.85rem; color:var(--text-muted);">Appointment ID</span>
        <div class="confirm-id">{{ appt.appointment_code }}</div>
      </div>

      <div class="confirm-actions">
        <a href="{{ url_for('appointments_page', email=appt.email) }}" class="btn btn-primary">View Appointment</a>
        <a href="{{ url_for('home') }}" class="btn btn-secondary">Back to Home</a>
      </div>
    </div>
  </div>
</section>

{% endblock %}

Writing templates/confirmation.html


### `templates/appointments.html`

In [12]:
%%writefile templates/appointments.html
{% extends "base.html" %}
{% block title %}My Appointments — Q-Schedule{% endblock %}
{% block content %}

<section class="section section-tight section-soft">
  <div class="container">
    <span class="eyebrow">My appointments</span>
    <h1 style="font-size:2.1rem;">Manage your appointments</h1>
    <p style="max-width:560px;">Look up your appointments using the email address you booked with.</p>
  </div>
</section>

<section class="section">
  <div class="container">
    <form class="email-lookup" id="lookupForm">
      <input type="email" id="lookupEmail" placeholder="you@example.com" value="{{ searched_email or '' }}" required>
      <button type="submit" class="btn btn-primary">Find My Appointments</button>
    </form>

    {% if searched_email %}
      {% if appointments %}
      <div class="grid grid-2" id="appointmentsGrid">
        {% for a in appointments %}
        <div class="card appt-card" data-code="{{ a.appointment_code }}" data-doctor-id="{{ a.doctor_id }}">
          <div class="left">
            <div class="doctor-avatar" style="width:52px;height:52px;">
              <img src="{{ url_for('static', filename='img/' + a.avatar_file) }}" alt="Portrait of {{ a.doctor_name }}">
            </div>
            <div class="details">
              <strong>{{ a.doctor_name }}</strong>
              <span class="dept">{{ a.department_name }}</span>
              <div class="when">{{ a.appointment_date }} · {{ a.slot_time }} · {{ a.duration_minutes }} min</div>
            </div>
          </div>
          <div style="display:flex; flex-direction:column; align-items:flex-end; gap:10px;">
            <span class="status-pill {{ 'confirmed' if a.status == 'Confirmed' else 'cancelled' }}">{{ a.status }}</span>
            {% if a.status == 'Confirmed' %}
            <div class="appt-actions">
              <button type="button" class="btn btn-secondary btn-sm" data-action="reschedule">Reschedule</button>
              <button type="button" class="btn btn-ghost btn-sm" data-action="cancel">Cancel</button>
            </div>
            {% endif %}
          </div>
          <div class="reschedule-panel" hidden style="width:100%; margin-top:16px; border-top:1px solid var(--border-soft); padding-top:16px;">
            <div class="form-grid">
              <div class="form-field">
                <label>New date</label>
                <input type="date" class="rs-date" min="{{ '' }}">
              </div>
              <div class="form-field">
                <label>New time</label>
                <select class="rs-time"><option value="">Select a date first</option></select>
              </div>
            </div>
            <button type="button" class="btn btn-primary btn-sm" data-action="save-reschedule">Save New Time</button>
          </div>
        </div>
        {% endfor %}
      </div>
      {% else %}
      <div class="empty-state">
        <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="3" y="4" width="18" height="18" rx="3"/><path d="M16 2v4M8 2v4M3 10h18"/></svg>
        <p>No appointments found for this email yet.</p>
        <a href="{{ url_for('book_page') }}" class="btn btn-primary" style="margin-top:10px;">Book an Appointment</a>
      </div>
      {% endif %}
    {% endif %}
  </div>
</section>

{% endblock %}

{% block scripts %}
<script src="{{ url_for('static', filename='js/appointments.js') }}"></script>
{% endblock %}

Writing templates/appointments.html


### `static/css/style.css`

In [13]:
%%writefile static/css/style.css
/* =========================================================================
   Q-SCHEDULE — Design tokens
   Light, clean, premium hospital platform. No dark backgrounds, no neon.
   ========================================================================= */

:root {
  /* --- Color: base palette ------------------------------------------- */
  --white:           #FFFFFF;
  --bg-soft-blue:     #F2F8FC;   /* very light blue section backgrounds  */
  --bg-soft-teal:      #EFFAF8;   /* light teal section backgrounds       */
  --medical-blue:      #2569B0;   /* soft medical blue — primary actions  */
  --medical-blue-dark: #184E86;
  --medical-blue-tint: #E7F1FB;
  --teal:              #17998F;   /* light teal accent                    */
  --teal-tint:         #E3F6F3;
  --lavender:          #8478C8;   /* small accents only                   */
  --lavender-tint:     #F1EEFB;
  --text-dark:         #24303F;   /* dark gray, not black                 */
  --text-muted:        #667487;
  --text-faint:        #94A2B5;
  --border:            #E4EAF2;
  --border-soft:       #EEF2F8;
  --success:           #1E9E6B;
  --success-tint:      #E7F8F0;
  --danger:            #D0483F;
  --danger-tint:       #FBEBEA;
  --shadow-sm: 0 1px 3px rgba(20, 45, 80, 0.05), 0 1px 2px rgba(20, 45, 80, 0.04);
  --shadow-md: 0 8px 24px rgba(20, 45, 80, 0.08);
  --shadow-lg: 0 16px 40px rgba(20, 45, 80, 0.12);
  --radius-sm: 10px;
  --radius-md: 16px;
  --radius-lg: 22px;
  --radius-pill: 999px;
  --max-width: 1200px;
}

* { box-sizing: border-box; }
html { scroll-behavior: smooth; }

body {
  margin: 0;
  background: var(--white);
  color: var(--text-dark);
  font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
  font-size: 16px;
  line-height: 1.6;
  -webkit-font-smoothing: antialiased;
}

h1, h2, h3, h4, h5 {
  font-family: 'Poppins', 'Inter', sans-serif;
  color: var(--text-dark);
  line-height: 1.25;
  margin: 0 0 0.5em 0;
  font-weight: 600;
}

h1 { font-size: 2.75rem; font-weight: 600; letter-spacing: -0.01em; }
h2 { font-size: 2rem; font-weight: 600; letter-spacing: -0.01em; }
h3 { font-size: 1.3rem; font-weight: 600; }
p  { margin: 0 0 1em 0; color: var(--text-muted); }
a  { color: inherit; text-decoration: none; }
ul { margin: 0; padding: 0; list-style: none; }
img { max-width: 100%; display: block; }
button { font-family: inherit; cursor: pointer; }

.container {
  max-width: var(--max-width);
  margin: 0 auto;
  padding: 0 28px;
}

.section { padding: 88px 0; }
.section-soft { background: var(--bg-soft-blue); }
.section-tight { padding: 56px 0; }

.eyebrow {
  color: var(--teal);
  font-weight: 600;
  font-size: 0.92rem;
  margin-bottom: 10px;
  display: inline-block;
}

.section-head { max-width: 640px; margin: 0 0 48px 0; }
.section-head.center { margin-left: auto; margin-right: auto; text-align: center; }
.section-head p { font-size: 1.08rem; }

/* --- Focus visibility (accessibility) ---------------------------------- */
a:focus-visible, button:focus-visible, input:focus-visible,
select:focus-visible, textarea:focus-visible {
  outline: 3px solid var(--medical-blue);
  outline-offset: 2px;
}

@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after { animation-duration: 0.001ms !important; transition-duration: 0.001ms !important; }
}

/* =========================================================================
   Buttons
   ========================================================================= */

.btn {
  display: inline-flex;
  align-items: center;
  justify-content: center;
  gap: 8px;
  padding: 14px 26px;
  border-radius: var(--radius-pill);
  font-weight: 600;
  font-size: 0.98rem;
  border: 1.5px solid transparent;
  transition: transform 0.15s ease, box-shadow 0.15s ease, background 0.15s ease;
  white-space: nowrap;
}
.btn:active { transform: translateY(1px); }

.btn-primary {
  background: var(--medical-blue);
  color: var(--white);
  box-shadow: 0 6px 18px rgba(37, 105, 176, 0.28);
}
.btn-primary:hover { background: var(--medical-blue-dark); box-shadow: 0 8px 22px rgba(37, 105, 176, 0.34); }

.btn-secondary {
  background: var(--white);
  color: var(--medical-blue);
  border-color: var(--border);
}
.btn-secondary:hover { border-color: var(--medical-blue); background: var(--medical-blue-tint); }

.btn-teal {
  background: var(--teal);
  color: var(--white);
  box-shadow: 0 6px 18px rgba(23, 153, 143, 0.28);
}
.btn-teal:hover { background: #128179; }

.btn-ghost {
  background: transparent;
  color: var(--text-muted);
  border-color: var(--border);
}
.btn-ghost:hover { color: var(--text-dark); border-color: var(--text-faint); }

.btn-block { width: 100%; }
.btn-sm { padding: 9px 16px; font-size: 0.88rem; }
.btn:disabled { opacity: 0.5; cursor: not-allowed; box-shadow: none; transform: none; }

/* =========================================================================
   Header / Navigation
   ========================================================================= */

.site-header {
  position: sticky;
  top: 0;
  z-index: 100;
  background: rgba(255, 255, 255, 0.92);
  backdrop-filter: blur(10px);
  border-bottom: 1px solid var(--border-soft);
}

.nav {
  max-width: var(--max-width);
  margin: 0 auto;
  padding: 16px 28px;
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 24px;
}

.brand {
  display: flex;
  align-items: center;
  gap: 10px;
  font-family: 'Poppins', sans-serif;
  font-weight: 700;
  font-size: 1.3rem;
  color: var(--text-dark);
}
.brand .brand-mark {
  width: 36px;
  height: 36px;
  border-radius: 10px;
  background: linear-gradient(135deg, var(--medical-blue), var(--teal));
  display: flex;
  align-items: center;
  justify-content: center;
  color: var(--white);
  flex-shrink: 0;
}
.brand .brand-mark svg { width: 20px; height: 20px; }
.brand .brand-sub {
  display: block;
  font-family: 'Inter', sans-serif;
  font-weight: 400;
  font-size: 0.72rem;
  color: var(--text-muted);
  letter-spacing: 0.01em;
}

.nav-links {
  display: flex;
  align-items: center;
  gap: 30px;
  flex-wrap: wrap;
}
.nav-links a {
  font-size: 0.96rem;
  font-weight: 500;
  color: var(--text-muted);
  padding: 6px 2px;
  border-bottom: 2px solid transparent;
  transition: color 0.15s ease, border-color 0.15s ease;
}
.nav-links a:hover, .nav-links a.active { color: var(--medical-blue); border-color: var(--medical-blue); }

.nav-actions { display: flex; align-items: center; gap: 14px; }

.nav-toggle { display: none; background: none; border: none; padding: 8px; }
.nav-toggle svg { width: 24px; height: 24px; color: var(--text-dark); }

.nav-actions .btn .short { display: none; }
@media (max-width: 480px) {
  .nav-actions .btn .long { display: none; }
  .nav-actions .btn .short { display: inline; }
}

@media (max-width: 920px) {
  .nav-links { position: absolute; top: 100%; left: 0; right: 0; background: var(--white);
    flex-direction: column; align-items: flex-start; padding: 18px 28px; gap: 16px;
    border-bottom: 1px solid var(--border-soft); box-shadow: var(--shadow-md); display: none; }
  .nav-links.open { display: flex; }
  .nav-toggle { display: block; }
}

/* =========================================================================
   Hero
   ========================================================================= */

.hero {
  padding: 64px 0 96px;
}
.hero-grid {
  display: grid;
  grid-template-columns: 1.05fr 0.95fr;
  gap: 56px;
  align-items: center;
}
.hero-copy .eyebrow { display: flex; align-items: center; gap: 8px; }
.hero-copy .eyebrow .dot { width: 7px; height: 7px; border-radius: 50%; background: var(--teal); display: inline-block; }
.hero-copy h1 { font-size: 3rem; }
.hero-copy p.lead { font-size: 1.15rem; max-width: 480px; }
.hero-actions { display: flex; gap: 14px; margin-top: 30px; flex-wrap: wrap; }

.hero-stats { display: flex; gap: 36px; margin-top: 46px; flex-wrap: wrap; }
.hero-stat .num { font-family: 'Poppins', sans-serif; font-size: 1.7rem; font-weight: 700; color: var(--medical-blue); }
.hero-stat .label { font-size: 0.86rem; color: var(--text-muted); }

.hero-visual { position: relative; }
.hero-photo {
  border-radius: var(--radius-lg);
  overflow: hidden;
  box-shadow: var(--shadow-lg);
  aspect-ratio: 4 / 4.6;
  background: var(--bg-soft-blue);
}
.hero-photo img { width: 100%; height: 100%; object-fit: cover; }

.hero-floating-card {
  position: absolute;
  background: var(--white);
  border-radius: var(--radius-md);
  box-shadow: var(--shadow-lg);
  padding: 16px 18px;
  display: flex;
  align-items: center;
  gap: 12px;
  max-width: 240px;
}
.hero-floating-card.top { top: -20px; right: -18px; }
.hero-floating-card.bottom { bottom: 24px; left: -28px; }
.hero-floating-card .icon-badge { width: 40px; height: 40px; border-radius: 10px; display: flex; align-items: center; justify-content: center; flex-shrink: 0; }
.hero-floating-card .icon-badge.blue { background: var(--medical-blue-tint); color: var(--medical-blue); }
.hero-floating-card .icon-badge.teal { background: var(--teal-tint); color: var(--teal); }
.hero-floating-card .icon-badge svg { width: 20px; height: 20px; }
.hero-floating-card strong { display: block; font-size: 0.92rem; color: var(--text-dark); }
.hero-floating-card span { font-size: 0.8rem; color: var(--text-muted); }

@media (max-width: 920px) {
  .hero-grid { grid-template-columns: 1fr; }
  .hero-visual { order: -1; max-width: 420px; margin: 0 auto; }
  .hero-copy h1 { font-size: 2.3rem; }
  .hero-floating-card { display: none; }
}

/* =========================================================================
   Cards — features / doctors / departments
   ========================================================================= */

.grid {
  display: grid;
  gap: 24px;
}
.grid-4 { grid-template-columns: repeat(4, 1fr); }
.grid-3 { grid-template-columns: repeat(3, 1fr); }
.grid-2 { grid-template-columns: repeat(2, 1fr); }
@media (max-width: 980px) { .grid-4 { grid-template-columns: repeat(2, 1fr); } .grid-3 { grid-template-columns: repeat(2, 1fr); } }
@media (max-width: 620px) { .grid-4, .grid-3, .grid-2 { grid-template-columns: 1fr; } }

.card {
  background: var(--white);
  border: 1px solid var(--border-soft);
  border-radius: var(--radius-md);
  box-shadow: var(--shadow-sm);
  transition: box-shadow 0.2s ease, transform 0.2s ease;
}

.feature-card {
  padding: 30px 26px;
}
.feature-card .icon-badge {
  width: 48px; height: 48px; border-radius: 12px;
  display: flex; align-items: center; justify-content: center;
  margin-bottom: 18px;
}
.feature-card .icon-badge svg { width: 24px; height: 24px; }
.feature-card h3 { margin-bottom: 8px; font-size: 1.1rem; }
.feature-card p { margin: 0; font-size: 0.95rem; }
.icon-badge.blue   { background: var(--medical-blue-tint); color: var(--medical-blue); }
.icon-badge.teal   { background: var(--teal-tint); color: var(--teal); }
.icon-badge.lavender { background: var(--lavender-tint); color: var(--lavender); }

/* Doctor cards */
.doctor-card {
  padding: 26px;
  display: flex;
  flex-direction: column;
  gap: 14px;
}
.doctor-card:hover { box-shadow: var(--shadow-md); transform: translateY(-2px); }
.doctor-card-top { display: flex; gap: 14px; align-items: center; }
.doctor-avatar {
  width: 64px; height: 64px; border-radius: 50%;
  overflow: hidden; flex-shrink: 0; background: var(--bg-soft-blue);
  border: 2px solid var(--white); box-shadow: 0 0 0 1px var(--border-soft);
}
.doctor-avatar img { width: 100%; height: 100%; object-fit: cover; }
.doctor-name { font-family: 'Poppins', sans-serif; font-weight: 600; font-size: 1.05rem; margin: 0; }
.doctor-specialty { color: var(--medical-blue); font-size: 0.88rem; font-weight: 500; }
.doctor-meta { display: flex; gap: 14px; flex-wrap: wrap; font-size: 0.85rem; color: var(--text-muted); }
.doctor-meta span { display: flex; align-items: center; gap: 5px; }
.doctor-meta svg { width: 15px; height: 15px; color: var(--text-faint); }

.availability-badge {
  display: inline-flex; align-items: center; gap: 6px;
  font-size: 0.8rem; font-weight: 600; color: var(--success);
  background: var(--success-tint); padding: 5px 11px; border-radius: var(--radius-pill);
  width: fit-content;
}
.availability-badge .dot { width: 6px; height: 6px; border-radius: 50%; background: var(--success); }

.next-slot { font-size: 0.88rem; color: var(--text-muted); }
.next-slot strong { color: var(--text-dark); font-family: 'Poppins', sans-serif; }

.doctor-filter-bar {
  display: flex; gap: 10px; flex-wrap: wrap; margin-bottom: 36px;
}
.filter-chip {
  padding: 9px 18px; border-radius: var(--radius-pill); border: 1.5px solid var(--border);
  font-size: 0.9rem; font-weight: 500; color: var(--text-muted); background: var(--white);
  transition: all 0.15s ease;
}
.filter-chip:hover { border-color: var(--medical-blue); color: var(--medical-blue); }
.filter-chip.active { background: var(--medical-blue); border-color: var(--medical-blue); color: var(--white); }

/* Department cards */
.dept-card { padding: 28px; text-align: left; }
.dept-card .dept-icon {
  width: 54px; height: 54px; border-radius: 14px; margin-bottom: 18px;
  display: flex; align-items: center; justify-content: center;
  background: var(--medical-blue-tint); color: var(--medical-blue);
}
.dept-card .dept-icon svg { width: 27px; height: 27px; }
.dept-card h3 { margin-bottom: 6px; }
.dept-card p { font-size: 0.94rem; margin-bottom: 16px; }
.dept-card .dept-footer { display: flex; align-items: center; justify-content: space-between; padding-top: 14px; border-top: 1px solid var(--border-soft); }
.dept-card .doctor-count { font-size: 0.85rem; color: var(--text-muted); }
.dept-card .doctor-count strong { color: var(--text-dark); }

/* =========================================================================
   How it works / steps
   ========================================================================= */

.steps-row { display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; }
@media (max-width: 900px) { .steps-row { grid-template-columns: repeat(2, 1fr); } }
@media (max-width: 560px) { .steps-row { grid-template-columns: 1fr; } }
.step-item { position: relative; padding: 28px 22px; background: var(--white); border-radius: var(--radius-md); border: 1px solid var(--border-soft); }
.step-num {
  width: 34px; height: 34px; border-radius: 50%; background: var(--medical-blue);
  color: var(--white); font-family: 'Poppins', sans-serif; font-weight: 700; font-size: 0.95rem;
  display: flex; align-items: center; justify-content: center; margin-bottom: 16px;
}
.step-item h3 { font-size: 1rem; margin-bottom: 6px; }
.step-item p { font-size: 0.9rem; margin: 0; }

/* =========================================================================
   CTA band
   ========================================================================= */

.cta-band {
  background: linear-gradient(120deg, var(--medical-blue), var(--teal));
  border-radius: var(--radius-lg);
  padding: 56px 48px;
  color: var(--white);
  display: flex; align-items: center; justify-content: space-between; gap: 28px;
  flex-wrap: wrap;
}
.cta-band h2 { color: var(--white); margin-bottom: 6px; }
.cta-band p { color: rgba(255,255,255,0.88); margin: 0; }
.cta-band .btn-primary { background: var(--white); color: var(--medical-blue-dark); box-shadow: none; }
.cta-band .btn-primary:hover { background: #F3F7FC; }

/* =========================================================================
   Footer
   ========================================================================= */

.site-footer { background: var(--bg-soft-blue); padding: 64px 0 28px; margin-top: 40px; }
.footer-grid { display: grid; grid-template-columns: 1.4fr 1fr 1fr 1fr; gap: 36px; padding-bottom: 40px; }
@media (max-width: 780px) { .footer-grid { grid-template-columns: 1fr 1fr; } }
.footer-grid h4 { font-size: 0.92rem; margin-bottom: 14px; color: var(--text-dark); }
.footer-grid ul li { margin-bottom: 10px; }
.footer-grid ul li a { font-size: 0.9rem; color: var(--text-muted); }
.footer-grid ul li a:hover { color: var(--medical-blue); }
.footer-bottom { border-top: 1px solid var(--border); padding-top: 22px; display: flex; justify-content: space-between; flex-wrap: wrap; gap: 10px; font-size: 0.85rem; color: var(--text-faint); }

/* =========================================================================
   Booking wizard
   ========================================================================= */

.booking-shell { display: grid; grid-template-columns: 260px 1fr; gap: 40px; align-items: flex-start; }
@media (max-width: 900px) { .booking-shell { grid-template-columns: 1fr; } }

.wizard-nav { position: sticky; top: 96px; }
.wizard-step {
  display: flex; align-items: center; gap: 12px; padding: 12px 4px;
  color: var(--text-faint); font-weight: 500; font-size: 0.94rem;
}
.wizard-step .circle {
  width: 28px; height: 28px; border-radius: 50%; border: 2px solid var(--border);
  display: flex; align-items: center; justify-content: center; font-size: 0.78rem; font-weight: 600;
  flex-shrink: 0; color: var(--text-faint); background: var(--white);
}
.wizard-step.active { color: var(--text-dark); }
.wizard-step.active .circle { border-color: var(--medical-blue); color: var(--medical-blue); }
.wizard-step.done .circle { background: var(--medical-blue); border-color: var(--medical-blue); color: var(--white); }
.wizard-step.done { color: var(--text-dark); }

.booking-panel {
  background: var(--white); border: 1px solid var(--border-soft); border-radius: var(--radius-lg);
  box-shadow: var(--shadow-sm); padding: 40px;
}
@media (max-width: 560px) { .booking-panel { padding: 24px; } }
.booking-panel h2 { margin-bottom: 6px; }
.booking-panel .step-sub { margin-bottom: 30px; }

.form-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }
@media (max-width: 560px) { .form-grid { grid-template-columns: 1fr; } }
.form-field { display: flex; flex-direction: column; gap: 7px; }
.form-field.full { grid-column: 1 / -1; }
.form-field label { font-size: 0.88rem; font-weight: 600; color: var(--text-dark); }
.form-field input, .form-field select {
  padding: 13px 15px; border: 1.5px solid var(--border); border-radius: var(--radius-sm);
  font-size: 0.96rem; font-family: inherit; color: var(--text-dark); background: var(--white);
}
.form-field input:focus, .form-field select:focus { border-color: var(--medical-blue); outline: none; }
.field-error { font-size: 0.8rem; color: var(--danger); min-height: 16px; }

.option-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 14px; }
@media (max-width: 700px) { .option-grid { grid-template-columns: repeat(2, 1fr); } }
@media (max-width: 480px) { .option-grid { grid-template-columns: 1fr; } }

.option-card {
  border: 1.5px solid var(--border); border-radius: var(--radius-md); padding: 18px;
  cursor: pointer; transition: all 0.15s ease; background: var(--white); text-align: left;
}
.option-card:hover { border-color: var(--medical-blue); }
.option-card.selected { border-color: var(--medical-blue); background: var(--medical-blue-tint); box-shadow: 0 0 0 1px var(--medical-blue); }
.option-card .opt-icon { width: 40px; height: 40px; border-radius: 10px; background: var(--bg-soft-blue); color: var(--medical-blue); display: flex; align-items: center; justify-content: center; margin-bottom: 12px; }
.option-card .opt-icon svg { width: 20px; height: 20px; }
.option-card strong { display: block; font-size: 0.96rem; margin-bottom: 3px; }
.option-card span { font-size: 0.82rem; color: var(--text-muted); }

.doctor-select-card {
  display: flex; align-items: center; gap: 14px; border: 1.5px solid var(--border);
  border-radius: var(--radius-md); padding: 16px; cursor: pointer; transition: all 0.15s ease;
}
.doctor-select-card:hover { border-color: var(--medical-blue); }
.doctor-select-card.selected { border-color: var(--medical-blue); background: var(--medical-blue-tint); }
.doctor-select-card .doctor-avatar { width: 48px; height: 48px; }
.doctor-select-card .info strong { display: block; font-size: 0.95rem; }
.doctor-select-card .info span { font-size: 0.82rem; color: var(--text-muted); }

input[type="date"] {
  padding: 13px 15px; border: 1.5px solid var(--border); border-radius: var(--radius-sm);
  font-size: 0.96rem; font-family: inherit; color: var(--text-dark); width: 100%; max-width: 320px;
}

.slot-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-top: 8px; }
@media (max-width: 700px) { .slot-grid { grid-template-columns: repeat(3, 1fr); } }
@media (max-width: 480px) { .slot-grid { grid-template-columns: repeat(2, 1fr); } }

.slot-card {
  border: 1.5px solid var(--border); border-radius: var(--radius-sm); padding: 13px 10px;
  text-align: center; font-weight: 600; font-size: 0.9rem; color: var(--text-dark);
  background: var(--white); transition: all 0.15s ease;
}
.slot-card:hover:not(.booked) { border-color: var(--medical-blue); color: var(--medical-blue); }
.slot-card.selected { background: var(--medical-blue); border-color: var(--medical-blue); color: var(--white); }
.slot-card.booked { background: var(--border-soft); color: var(--text-faint); cursor: not-allowed; text-decoration: line-through; }

.optimize-banner {
  display: flex; align-items: center; justify-content: space-between; gap: 20px; flex-wrap: wrap;
  background: var(--lavender-tint); border: 1px solid #E2DCF7; border-radius: var(--radius-md);
  padding: 20px 24px; margin: 28px 0;
}
.optimize-banner .txt strong { display: block; font-size: 0.98rem; color: var(--text-dark); margin-bottom: 3px; }
.optimize-banner .txt span { font-size: 0.86rem; color: var(--text-muted); }

.smart-card {
  border: 1px solid var(--border-soft); border-radius: var(--radius-md); padding: 22px 24px;
  background: var(--bg-soft-blue); margin: 24px 0;
}
.smart-card h4 { font-size: 0.95rem; margin-bottom: 14px; display: flex; align-items: center; gap: 8px; }
.smart-card h4 svg { width: 18px; height: 18px; color: var(--lavender); }
.smart-steps li { display: flex; align-items: center; gap: 10px; font-size: 0.88rem; color: var(--text-muted); padding: 5px 0; opacity: 0; animation: fadeInStep 0.35s ease forwards; }
.smart-steps li svg { width: 16px; height: 16px; color: var(--success); flex-shrink: 0; }
@keyframes fadeInStep { from { opacity: 0; transform: translateY(4px); } to { opacity: 1; transform: translateY(0); } }

.recommend-card {
  display: flex; align-items: center; gap: 20px; background: var(--white);
  border: 1.5px solid var(--medical-blue); border-radius: var(--radius-md); padding: 22px 24px; margin-top: 18px;
}
.recommend-card .time-badge {
  background: var(--medical-blue); color: var(--white); border-radius: var(--radius-md);
  padding: 14px 18px; text-align: center; font-family: 'Poppins', sans-serif; min-width: 92px;
}
.recommend-card .time-badge .t { font-size: 1.3rem; font-weight: 700; display: block; }
.recommend-card .time-badge .d { font-size: 0.72rem; opacity: 0.85; }
.recommend-card .info strong { display: block; font-size: 1.02rem; }
.recommend-card .info span { font-size: 0.86rem; color: var(--text-muted); }
.qaoa-tag {
  display: inline-flex; align-items: center; gap: 6px; font-size: 0.74rem; font-weight: 700;
  color: var(--lavender); background: var(--lavender-tint); padding: 4px 10px; border-radius: var(--radius-pill);
  margin-top: 8px; letter-spacing: 0.01em;
}

.summary-list { border: 1px solid var(--border-soft); border-radius: var(--radius-md); overflow: hidden; margin-bottom: 24px; }
.summary-row { display: flex; justify-content: space-between; padding: 14px 20px; font-size: 0.94rem; }
.summary-row:nth-child(odd) { background: var(--bg-soft-blue); }
.summary-row .k { color: var(--text-muted); }
.summary-row .v { font-weight: 600; color: var(--text-dark); }

.wizard-actions { display: flex; justify-content: space-between; margin-top: 32px; gap: 14px; }

/* =========================================================================
   Confirmation page
   ========================================================================= */

.confirm-wrap { max-width: 560px; margin: 0 auto; text-align: center; }
.confirm-check {
  width: 76px; height: 76px; border-radius: 50%; background: var(--success-tint); color: var(--success);
  display: flex; align-items: center; justify-content: center; margin: 0 auto 22px;
}
.confirm-check svg { width: 38px; height: 38px; }
.confirm-id { display: inline-block; background: var(--bg-soft-blue); color: var(--medical-blue); font-weight: 700;
  padding: 8px 18px; border-radius: var(--radius-pill); font-family: 'Poppins', sans-serif; font-size: 0.9rem; margin-top: 6px; }
.confirm-actions { display: flex; gap: 14px; justify-content: center; margin-top: 30px; flex-wrap: wrap; }

/* =========================================================================
   My appointments
   ========================================================================= */

.appt-card { padding: 24px; display: flex; align-items: center; justify-content: space-between; gap: 20px; flex-wrap: wrap; }
.appt-card .left { display: flex; gap: 16px; align-items: center; }
.appt-card .details strong { display: block; font-size: 1.02rem; }
.appt-card .details .dept { color: var(--medical-blue); font-size: 0.86rem; font-weight: 500; }
.appt-card .when { font-size: 0.88rem; color: var(--text-muted); margin-top: 4px; }
.status-pill { font-size: 0.78rem; font-weight: 700; padding: 5px 12px; border-radius: var(--radius-pill); }
.status-pill.confirmed { background: var(--success-tint); color: var(--success); }
.status-pill.cancelled { background: var(--danger-tint); color: var(--danger); }
.appt-actions { display: flex; gap: 10px; }

.email-lookup { display: flex; gap: 12px; max-width: 480px; margin-bottom: 40px; flex-wrap: wrap; }
.email-lookup input { flex: 1; min-width: 220px; padding: 13px 16px; border: 1.5px solid var(--border); border-radius: var(--radius-sm); font-family: inherit; font-size: 0.95rem; }

.empty-state { text-align: center; padding: 60px 20px; color: var(--text-muted); }
.empty-state svg { width: 52px; height: 52px; color: var(--text-faint); margin-bottom: 14px; }

/* =========================================================================
   Quantum details page ("How Quantum Scheduling Works")
   ========================================================================= */

.quantum-hero { background: var(--bg-soft-blue); padding: 72px 0 56px; }
.quantum-flow { display: flex; flex-direction: column; gap: 0; max-width: 760px; margin: 0 auto; }
.flow-step { display: flex; gap: 22px; padding: 26px 0; }
.flow-step .flow-icon {
  width: 52px; height: 52px; border-radius: 14px; background: var(--medical-blue-tint); color: var(--medical-blue);
  display: flex; align-items: center; justify-content: center; flex-shrink: 0;
}
.flow-step .flow-icon svg { width: 26px; height: 26px; }
.flow-step .flow-body h3 { margin-bottom: 6px; }
.flow-step .flow-body p { margin: 0; }
.flow-connector { width: 2px; height: 28px; background: var(--border); margin-left: 25px; }

.circuit-panel { background: var(--white); border: 1px solid var(--border-soft); border-radius: var(--radius-lg); padding: 32px; text-align: center; box-shadow: var(--shadow-sm); }
.circuit-panel img { max-width: 100%; border-radius: var(--radius-sm); }
.circuit-meta { display: flex; justify-content: center; gap: 30px; margin-top: 20px; flex-wrap: wrap; }
.circuit-meta div { text-align: center; }
.circuit-meta .num { font-family: 'Poppins', sans-serif; font-weight: 700; font-size: 1.3rem; color: var(--medical-blue); }
.circuit-meta .lbl { font-size: 0.8rem; color: var(--text-muted); }

.tech-tags { display: flex; gap: 10px; flex-wrap: wrap; margin-top: 18px; }
.tech-tag { font-size: 0.8rem; font-weight: 600; padding: 6px 14px; border-radius: var(--radius-pill); background: var(--teal-tint); color: var(--teal); }

.note-box { background: var(--lavender-tint); border: 1px solid #E2DCF7; border-radius: var(--radius-md); padding: 18px 22px; font-size: 0.92rem; color: var(--text-dark); margin-top: 28px; }
.note-box strong { color: var(--lavender); }

/* =========================================================================
   Misc utility
   ========================================================================= */

.text-center { text-align: center; }
.mt-0 { margin-top: 0; }
.spinner {
  width: 18px; height: 18px; border-radius: 50%; border: 2.5px solid rgba(255,255,255,0.4);
  border-top-color: var(--white); animation: spin 0.7s linear infinite; display: inline-block;
}
.spinner.dark { border: 2.5px solid var(--border); border-top-color: var(--medical-blue); }
@keyframes spin { to { transform: rotate(360deg); } }

.toast {
  position: fixed; bottom: 26px; left: 50%; transform: translateX(-50%) translateY(20px);
  background: var(--text-dark); color: var(--white); padding: 14px 22px; border-radius: var(--radius-sm);
  font-size: 0.9rem; box-shadow: var(--shadow-lg); opacity: 0; pointer-events: none;
  transition: all 0.25s ease; z-index: 999;
}
.toast.show { opacity: 1; transform: translateX(-50%) translateY(0); }
.toast.error { background: var(--danger); }

Writing static/css/style.css


### `static/js/booking.js`

In [14]:
%%writefile static/js/booking.js
(function () {
  "use strict";

  const state = {
    step: 1,
    department: null,   // {id, name}
    doctor: null,        // full doctor object
    date: document.getElementById("apptDate")?.value || "",
    time: null,
    duration: 30,
    optimizedByQaoa: false,
  };

  const panels = document.querySelectorAll(".wizard-panel");
  const navSteps = document.querySelectorAll("[data-nav-step]");

  function goToStep(n) {
    state.step = n;
    panels.forEach((p) => (p.hidden = Number(p.dataset.panel) !== n));
    navSteps.forEach((el) => {
      const s = Number(el.dataset.navStep);
      el.classList.toggle("active", s === n);
      el.classList.toggle("done", s < n);
    });
    if (n === 5) loadAvailability();
    if (n === 6) renderSummary();
    window.scrollTo({ top: document.querySelector(".booking-shell").offsetTop - 90, behavior: "smooth" });
  }

  document.querySelectorAll("[data-next]").forEach((btn) => {
    btn.addEventListener("click", () => {
      if (!validateStep(state.step)) return;
      goToStep(Number(btn.dataset.next));
    });
  });
  document.querySelectorAll("[data-prev]").forEach((btn) => {
    btn.addEventListener("click", () => goToStep(Number(btn.dataset.prev)));
  });

  function validateStep(step) {
    if (step === 1) {
      const fields = [
        ["p_name", (v) => v.trim().length > 1, "Please enter the patient's name."],
        ["p_age", (v) => v && Number(v) > 0 && Number(v) < 121, "Enter a valid age."],
        ["p_gender", (v) => !!v, "Please select a gender."],
        ["p_phone", (v) => v.trim().replace(/\D/g, "").length >= 7, "Enter a valid phone number."],
        ["p_email", (v) => /^[^\s@]+@[^\s@]+\.[^\s@]+$/.test(v), "Enter a valid email address."],
      ];
      let ok = true;
      fields.forEach(([id, test, msg]) => {
        const el = document.getElementById(id);
        const errEl = el.closest(".form-field").querySelector(".field-error");
        if (!test(el.value)) {
          errEl.textContent = msg;
          ok = false;
        } else {
          errEl.textContent = "";
        }
      });
      return ok;
    }
    if (step === 5 && !state.time) {
      showToast("Please choose a time slot to continue.", true);
      return false;
    }
    return true;
  }

  // --- Step 2: department selection ----------------------------------
  document.querySelectorAll("#departmentGrid .option-card").forEach((card) => {
    card.addEventListener("click", () => {
      document.querySelectorAll("#departmentGrid .option-card").forEach((c) => c.classList.remove("selected"));
      card.classList.add("selected");
      state.department = { id: Number(card.dataset.deptId), name: card.dataset.deptName };
      state.doctor = null;
      document.getElementById("toStep3").disabled = false;
    });
  });

  if (window.QS_PRESELECT_DEPARTMENT) {
    const card = document.querySelector(`#departmentGrid .option-card[data-dept-id="${window.QS_PRESELECT_DEPARTMENT}"]`);
    if (card) card.click();
  }

  // --- Step 3: doctor selection ----------------------------------------
  function renderDoctors() {
    const grid = document.getElementById("doctorGrid");
    document.getElementById("deptLabel").textContent = state.department ? state.department.name : "this department";
    const doctors = (window.QS_DOCTORS || []).filter(
      (d) => !state.department || d.department_id === state.department.id
    );
    grid.innerHTML = "";
    if (!doctors.length) {
      grid.innerHTML = '<p style="color:var(--text-faint);">No doctors found for this department.</p>';
      return;
    }
    doctors.forEach((doc) => {
      const div = document.createElement("div");
      div.className = "doctor-select-card";
      div.dataset.doctorId = doc.id;
      div.innerHTML = `
        <div class="doctor-avatar"><img src="/static/img/${doc.avatar_file}" alt="Portrait of ${doc.name}"></div>
        <div class="info">
          <strong>${doc.name}</strong>
          <span>${doc.department_name} · ${doc.experience_years} yrs experience</span>
        </div>`;
      div.addEventListener("click", () => {
        document.querySelectorAll(".doctor-select-card").forEach((c) => c.classList.remove("selected"));
        div.classList.add("selected");
        state.doctor = doc;
        document.getElementById("toStep4").disabled = false;
      });
      grid.appendChild(div);
    });

    if (window.QS_PRESELECT_DOCTOR) {
      const match = grid.querySelector(`[data-doctor-id="${window.QS_PRESELECT_DOCTOR}"]`);
      if (match) match.click();
    }
  }

  document.querySelectorAll('[data-next="3"]').forEach((btn) =>
    btn.addEventListener("click", () => {
      if (state.department) renderDoctors();
    })
  );

  // --- Step 4: date -------------------------------------------------------
  document.getElementById("apptDate").addEventListener("change", (e) => {
    state.date = e.target.value;
  });

  // --- Step 5: time / availability / QAOA optimize -------------------------
  async function loadAvailability() {
    const grid = document.getElementById("slotGrid");
    if (!state.doctor || !state.date) {
      grid.innerHTML = '<p style="grid-column:1/-1; color:var(--text-faint);">Select a doctor and date first.</p>';
      return;
    }
    grid.innerHTML = '<p style="grid-column:1/-1; color:var(--text-faint);">Loading available slots…</p>';
    try {
      const res = await fetch(`/api/availability?doctor=${state.doctor.id}&date=${state.date}`);
      const data = await res.json();
      grid.innerHTML = "";
      data.slots.forEach((slot) => {
        const btn = document.createElement("button");
        btn.type = "button";
        btn.className = "slot-card" + (slot.available ? "" : " booked") + (state.time === slot.time ? " selected" : "");
        btn.textContent = formatTime(slot.time) + (slot.available ? "" : " · Booked");
        btn.disabled = !slot.available;
        btn.addEventListener("click", () => selectTime(slot.time, false));
        grid.appendChild(btn);
      });
    } catch (err) {
      grid.innerHTML = '<p style="grid-column:1/-1; color:var(--danger);">Could not load availability. Please try again.</p>';
    }
  }

  function selectTime(time, fromOptimizer) {
    state.time = time;
    state.optimizedByQaoa = !!fromOptimizer;
    document.querySelectorAll(".slot-card").forEach((c) => {
      c.classList.toggle("selected", c.textContent.startsWith(formatTime(time)));
    });
    document.getElementById("toStep6").disabled = false;
  }

  function formatTime(t) {
    const [h, m] = t.split(":").map(Number);
    const period = h >= 12 ? "PM" : "AM";
    const hour12 = h % 12 === 0 ? 12 : h % 12;
    return `${hour12}:${String(m).padStart(2, "0")} ${period}`;
  }

  document.getElementById("optimizeBtn").addEventListener("click", async () => {
    if (!state.doctor || !state.date) {
      showToast("Please select a doctor and date first.", true);
      return;
    }
    const smartCard = document.getElementById("smartCard");
    const recommendCard = document.getElementById("recommendCard");
    recommendCard.innerHTML = "";
    smartCard.hidden = false;
    smartCard.className = "smart-card";
    smartCard.innerHTML = `
      <h4>
        <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 2v4M12 18v4M4.9 4.9l2.8 2.8M16.3 16.3l2.8 2.8M2 12h4M18 12h4M4.9 19.1l2.8-2.8M16.3 7.7l2.8-2.8"/></svg>
        Smart Scheduling
      </h4>
      <ul class="smart-steps" id="smartSteps"></ul>`;

    const stageLabels = [
      "Checking doctor availability",
      "Checking existing appointments",
      "Building optimization problem",
      "Finding the best fit for you",
      "Finding best available slot",
    ];
    const stepsEl = document.getElementById("smartSteps");
    stageLabels.forEach((label, i) => {
      const li = document.createElement("li");
      li.style.animationDelay = `${i * 0.25}s`;
      li.innerHTML = `<svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M20 6 9 17l-5-5"/></svg> ${label}`;
      stepsEl.appendChild(li);
    });

    const btn = document.getElementById("optimizeBtn");
    btn.disabled = true;
    const originalHtml = btn.innerHTML;
    btn.innerHTML = '<span class="spinner"></span> Optimizing…';

    try {
      const preferredTime = document.getElementById("preferredTime").value || null;
      const res = await fetch("/api/optimize", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ doctor_id: state.doctor.id, date: state.date, preferred_time: preferredTime }),
      });
      const data = await res.json();

      // Let the progress animation play out before showing the result.
      await new Promise((r) => setTimeout(r, stageLabels.length * 250 + 300));

      if (!data.success) {
        recommendCard.innerHTML = `<p style="color:var(--danger); margin-top:14px;">${data.message || "No suitable slot found."}</p>`;
        return;
      }

      recommendCard.innerHTML = `
        <div class="recommend-card">
          <div class="time-badge"><span class="t">${formatTime(data.recommended_time)}</span><span class="d">${data.duration_minutes} min</span></div>
          <div class="info">
            <strong>Recommended Time</strong>
            <span>${data.doctor_name} · ${data.department_name}</span>
            <p style="margin:10px 0 0; font-size:0.85rem; color:var(--text-muted);">${data.explanation}</p>
          </div>
          <button type="button" class="btn btn-primary btn-sm" id="useRecommended">Use this time</button>
        </div>`;

      document.getElementById("useRecommended").addEventListener("click", () => {
        selectTime(data.recommended_time, true);
        loadAvailability().then(() => selectTime(data.recommended_time, true));
        showToast("Recommended time selected.");
      });
    } catch (err) {
      recommendCard.innerHTML = `<p style="color:var(--danger); margin-top:14px;">Something went wrong while optimizing. Please try again.</p>`;
    } finally {
      btn.disabled = false;
      btn.innerHTML = originalHtml;
    }
  });

  // --- Step 6: summary + confirm ------------------------------------------
  function renderSummary() {
    const rows = [
      ["Patient", document.getElementById("p_name").value],
      ["Doctor", state.doctor ? state.doctor.name : "—"],
      ["Department", state.department ? state.department.name : "—"],
      ["Date", formatDate(state.date)],
      ["Time", state.time ? formatTime(state.time) : "—"],
      ["Duration", `${state.duration} minutes`],
    ];
    document.getElementById("summaryList").innerHTML = rows
      .map(([k, v]) => `<div class="summary-row"><span class="k">${k}</span><span class="v">${v}</span></div>`)
      .join("");
  }

  function formatDate(iso) {
    if (!iso) return "—";
    const d = new Date(iso + "T00:00:00");
    return d.toLocaleDateString("en-US", { day: "numeric", month: "long", year: "numeric" });
  }

  document.getElementById("bookingForm").addEventListener("submit", async (e) => {
    e.preventDefault();
    const confirmBtn = document.getElementById("confirmBtn");
    confirmBtn.disabled = true;
    const original = confirmBtn.textContent;
    confirmBtn.innerHTML = '<span class="spinner"></span> Confirming…';

    const payload = {
      name: document.getElementById("p_name").value,
      age: Number(document.getElementById("p_age").value),
      gender: document.getElementById("p_gender").value,
      phone: document.getElementById("p_phone").value,
      email: document.getElementById("p_email").value,
      department_id: state.department.id,
      doctor_id: state.doctor.id,
      date: state.date,
      time: state.time,
      duration_minutes: state.duration,
      optimized_by_qaoa: state.optimizedByQaoa,
    };

    try {
      const res = await fetch("/api/book", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(payload),
      });
      const data = await res.json();
      if (data.success) {
        window.location.href = `/confirmation/${data.appointment_code}`;
      } else {
        showToast(data.message || "Could not confirm appointment.", true);
        confirmBtn.disabled = false;
        confirmBtn.textContent = original;
        if (res.status === 409) goToStep(5);
      }
    } catch (err) {
      showToast("Network error. Please try again.", true);
      confirmBtn.disabled = false;
      confirmBtn.textContent = original;
    }
  });

  goToStep(1);
})();

Writing static/js/booking.js


### `static/js/appointments.js`

In [15]:
%%writefile static/js/appointments.js
(function () {
  "use strict";

  document.getElementById("lookupForm").addEventListener("submit", (e) => {
    e.preventDefault();
    const email = document.getElementById("lookupEmail").value.trim();
    if (!email) return;
    window.location.href = `/appointments?email=${encodeURIComponent(email)}`;
  });

  document.querySelectorAll(".appt-card").forEach((card) => {
    const code = card.dataset.code;
    const doctorId = card.dataset.doctorId;
    const panel = card.querySelector(".reschedule-panel");
    const dateInput = panel?.querySelector(".rs-date");
    const timeSelect = panel?.querySelector(".rs-time");

    card.querySelector('[data-action="reschedule"]')?.addEventListener("click", () => {
      panel.hidden = !panel.hidden;
      if (!panel.hidden && dateInput) {
        const today = new Date().toISOString().slice(0, 10);
        dateInput.min = today;
        if (!dateInput.value) dateInput.value = today;
        loadSlots();
      }
    });

    dateInput?.addEventListener("change", loadSlots);

    async function loadSlots() {
      timeSelect.innerHTML = '<option value="">Loading…</option>';
      try {
        const res = await fetch(`/api/availability?doctor=${doctorId}&date=${dateInput.value}`);
        const data = await res.json();
        timeSelect.innerHTML = "";
        data.slots.forEach((s) => {
          const opt = document.createElement("option");
          opt.value = s.time;
          opt.textContent = s.time + (s.available ? "" : " (Booked)");
          opt.disabled = !s.available;
          timeSelect.appendChild(opt);
        });
      } catch (err) {
        timeSelect.innerHTML = '<option value="">Could not load slots</option>';
      }
    }

    card.querySelector('[data-action="save-reschedule"]')?.addEventListener("click", async () => {
      const date = dateInput.value;
      const time = timeSelect.value;
      if (!date || !time) {
        showToast("Please choose a new date and time.", true);
        return;
      }
      try {
        const res = await fetch(`/api/appointments/${code}/reschedule`, {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify({ date, time }),
        });
        const data = await res.json();
        if (data.success) {
          showToast("Appointment rescheduled.");
          setTimeout(() => window.location.reload(), 700);
        } else {
          showToast(data.message || "Could not reschedule.", true);
        }
      } catch (err) {
        showToast("Network error. Please try again.", true);
      }
    });

    card.querySelector('[data-action="cancel"]')?.addEventListener("click", async () => {
      if (!window.confirm("Cancel this appointment?")) return;
      try {
        const res = await fetch(`/api/appointments/${code}/cancel`, { method: "POST" });
        const data = await res.json();
        if (data.success) {
          showToast("Appointment cancelled.");
          setTimeout(() => window.location.reload(), 700);
        } else {
          showToast(data.message || "Could not cancel.", true);
        }
      } catch (err) {
        showToast("Network error. Please try again.", true);
      }
    });
  });
})();

Writing static/js/appointments.js


### Doctor photos
Embedded as base64 so the notebook doesn't depend on any external image host.

In [16]:
import base64, pathlib
pathlib.Path('static/img').mkdir(parents=True, exist_ok=True)
photos = {
  'doctor-ananya-patel.jpg': '...',
  'doctor-arjun-rao.jpg': '...',
  'doctor-kiran-kumar.jpg': '...',
  'doctor-meera-reddy.jpg': '...',
  'doctor-priya-sharma.jpg': '...',
  'doctor-rahul-verma.jpg': '...',
}
for name, data in photos.items():
    pathlib.Path('static/img', name).write_bytes(base64.b64decode(data))
print(len(photos), 'doctor photos written')

6 doctor photos written


## 3. Run the QAOA scheduler on its own (quantum part)
This calls the same function the website uses and prints the recommended slot.

In [17]:
import sys, json
sys.path.insert(0, "/content/qschedule")
from datetime import date, timedelta
import database as db, quantum_scheduler as qs

db.init_db()
day = (date.today() + timedelta(days=2)).isoformat()
result = qs.run_qaoa_scheduler(doctor_id=1, appointment_date=day, preferred_time="10:15")
print(json.dumps(result, indent=2))

/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.13/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


{
  "success": true,
  "recommended_time": "10:30",
  "duration_minutes": 30,
  "doctor_name": "Dr. Priya Sharma",
  "department_name": "Cardiology",
  "candidates_considered": [
    {
      "time": "10:00",
      "available": true,
      "score": 0.9375
    },
    {
      "time": "10:30",
      "available": true,
      "score": 0.9375
    },
    {
      "time": "09:30",
      "available": true,
      "score": 0.8125
    },
    {
      "time": "11:00",
      "available": true,
      "score": 0.8125
    },
    {
      "time": "09:00",
      "available": true,
      "score": 0.6875
    },
    {
      "time": "11:30",
      "available": true,
      "score": 0.6875
    }
  ],
  "objective_value": -0.9375,
  "qaoa_reps": 2,
  "num_qubits": 6,
  "runtime_seconds": 24.674,
  "trace": [
    {
      "stage": "Checking doctor availability",
      "detail": "6 open slot(s) found"
    },
    {
      "stage": "Checking existing appointments",
      "detail": "0 slot(s) already booked"
    },
    {


## 4. Launch the website
Run this cell. The site loads inside the notebook, right below this cell. Keep the notebook running while you use it.

In [18]:
import sys, threading
sys.path.insert(0, "/content/qschedule")
from werkzeug.serving import make_server
import app as qapp

if "server" in globals():
    server.shutdown()
server = make_server("127.0.0.1", 5000, qapp.app, threaded=True)
threading.Thread(target=server.serve_forever, daemon=True).start()

from google.colab import output
output.serve_kernel_port_as_iframe(5000, height=900)

<IPython.core.display.Javascript object>

## 5. (Optional) Check every page loads
Run this after cell 4 if any page shows an error — it prints the exact Python traceback instead of a generic error page.

In [19]:
import traceback
qapp.app.config["PROPAGATE_EXCEPTIONS"] = True
c = qapp.app.test_client()
for path in ["/", "/doctors", "/doctors?department=1", "/departments", "/book", "/appointments"]:
    try:
        print(path, "->", c.get(path).status_code)
    except Exception:
        print(path, "FAILED")
        traceback.print_exc()

/ -> 200
/doctors -> 200
/doctors?department=1 -> 200
/departments -> 200
/book -> 200
/appointments -> 200
